In [1]:
import pandas as pd
import numpy as np
import json
import os

In [4]:
path = "../data/raw/"
file = "wuzzuf_data_engineer.json"
full_path = os.path.join(path, file)

In [8]:
with open(full_path, "r", encoding="utf-8") as f:
    data = json.load(f)

In [10]:
df = pd.json_normalize(data)

In [12]:
# Explore the data structure
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst few rows of jobs column:")
df['jobs'].head(2)

Shape: (1, 8)

Columns: ['jobs', 'metadata.keyword', 'metadata.total_jobs', 'metadata.pages_scraped', 'metadata.last_updated', 'metadata.updated_count', 'metadata.appended_count', 'metadata.source']

First few rows of jobs column:


0    [{'job_id': 'f3fi6wjnx9o9-middle-data-engineer...
Name: jobs, dtype: object

In [13]:
# ============================================================
# STEP 1: Load and Combine All JSON Files
# ============================================================
import pandas as pd
import numpy as np
import json
import os
import re
from typing import List, Dict, Any

# Define paths
path = "../data/raw/"
json_files = [
    "wuzzuf_data_engineer.json",
    "wuzzuf_data_analyst.json", 
    "wuzzuf_backend_developer.json",
    "wuzzuf_machine_learning.json"
]

def load_json_file(filepath: str) -> pd.DataFrame:
    """Load a JSON file and extract the jobs list"""
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    # Extract the jobs list from the wrapper
    jobs_list = data.get('jobs', [])
    
    # Flatten using json_normalize
    df = pd.json_normalize(jobs_list)
    return df

# Load and combine all files
all_dfs = []
for file in json_files:
    full_path = os.path.join(path, file)
    print(f"Loading: {file}")
    df = load_json_file(full_path)
    print(f"  → {len(df)} jobs loaded")
    all_dfs.append(df)

# Combine all DataFrames
combined_df = pd.concat(all_dfs, ignore_index=True)
print(f"\n✅ Total combined jobs: {len(combined_df)}")
print(f"Columns: {combined_df.columns.tolist()}")

Loading: wuzzuf_data_engineer.json
  → 21 jobs loaded
Loading: wuzzuf_data_analyst.json
  → 15 jobs loaded
Loading: wuzzuf_backend_developer.json
  → 16 jobs loaded
Loading: wuzzuf_machine_learning.json
  → 15 jobs loaded

✅ Total combined jobs: 67
Columns: ['job_id', 'title', 'company', 'location', 'job_type', 'experience', 'skills', 'posted_date', 'url', 'scraped_at', 'keyword']


In [14]:
# ============================================================
# STEP 2: Explore Data Quality Issues
# ============================================================
print("=" * 60)
print("DATA QUALITY EXPLORATION")
print("=" * 60)

# Sample data to understand structure
print("\n📋 Sample job record:")
print(combined_df.iloc[0])

print("\n\n📊 Experience field samples:")
print(combined_df['experience'].value_counts().head(10))

print("\n\n📊 Skills field samples (first 3):")
for i in range(3):
    print(f"\nJob {i}: {combined_df['skills'].iloc[i][:5]}...")

print("\n\n📊 Location field samples:")
print(combined_df['location'].value_counts().head(10))

print("\n\n📊 Company field samples:")
print(combined_df['company'].value_counts().head(10))

print("\n\n📊 Job Type field samples:")
print(combined_df['job_type'].value_counts())

DATA QUALITY EXPLORATION

📋 Sample job record:
job_id         f3fi6wjnx9o9-middle-data-engineer-vertex-techn...
title                                       Middle Data Engineer
company                                      Vertex Technologies
location                                            Cairo, Egypt
job_type                                               Full Time
experience                                       · 2+ Yrs of Exp
skills         [Vertex Technologies -, Experienced, · IT/Soft...
posted_date    Middle Data Engineer\nVertex Technologies - Ca...
url            https://wuzzuf.net/jobs/p/f3fi6wjnx9o9-middle-...
scraped_at                      2026-04-17T07:12:32.449244+00:00
keyword                                            data engineer
Name: 0, dtype: object


📊 Experience field samples:
experience
· 3 - 7 Yrs of Exp     15
· 0 - 3 Yrs of Exp      4
· 3 - 5 Yrs of Exp      4
· 2 - 3 Yrs of Exp      3
· 5+ Yrs of Exp         3
· 5 - 10 Yrs of Exp     3
· 3+ Yrs of Exp   

In [16]:
# ============================================================
# STEP 3: Build Cleaning Functions
# ============================================================

def clean_experience(exp_str: str) -> Dict[str, Any]:
    """
    Parse experience string and extract min/max years and experience level
    Examples: "· 2 - 5 Yrs of Exp" → min=2, max=5, level='mid'
              "· 5+ Yrs of Exp" → min=5, max=None, level='senior'
              "Not specified" → min=None, max=None, level=None
    """
    if pd.isna(exp_str) or not exp_str:
        return {'min_experience': None, 'max_experience': None, 'experience_level': None}
    
    # Remove bullet symbol and clean
    exp_clean = exp_str.replace('·', '').strip()
    
    # Pattern 1: "X - Y Yrs of Exp"
    range_match = re.search(r'(\d+)\s*-\s*(\d+)\s*Yrs?', exp_clean)
    if range_match:
        min_exp = int(range_match.group(1))
        max_exp = int(range_match.group(2))
        level = categorize_experience(min_exp, max_exp)
        return {'min_experience': min_exp, 'max_experience': max_exp, 'experience_level': level}
    
    # Pattern 2: "X+ Yrs of Exp"
    plus_match = re.search(r'(\d+)\+\s*Yrs?', exp_clean)
    if plus_match:
        min_exp = int(plus_match.group(1))
        return {'min_experience': min_exp, 'max_experience': None, 'experience_level': 'senior'}
    
    # Pattern 3: "X Yrs of Exp" (single number)
    single_match = re.search(r'(\d+)\s*Yrs?', exp_clean)
    if single_match:
        exp = int(single_match.group(1))
        return {'min_experience': exp, 'max_experience': exp, 'experience_level': categorize_experience(exp, exp)}
    
    return {'min_experience': None, 'max_experience': None, 'experience_level': None}


def categorize_experience(min_exp: int, max_exp: int) -> str:
    """Categorize experience level based on years"""
    avg_exp = (min_exp + max_exp) / 2 if max_exp else min_exp
    
    if avg_exp <= 2:
        return 'junior'
    elif avg_exp <= 5:
        return 'mid'
    elif avg_exp <= 8:
        return 'senior'
    else:
        return 'executive'


def clean_skills(skills_list: List[str]) -> List[str]:
    """
    Clean skills list by removing noise:
    - Company names (ending with "-")
    - "Experienced", "Entry Level", "Manager"
    - Bullet symbols "·"
    - Category tags like "IT/Software Development"
    """
    if skills_list is None or len(skills_list) == 0:
        return []
    
    noise_patterns = [
        r'^Experienced$',
        r'^Entry Level$',
        r'^Manager$',
        r'^·\s*',  # Bullet prefix
        r'.*\s*-\s*$',  # Company name suffix
        r'^IT/Software Development$',
        r'^Analyst/Research$',
        r'^Engineering$',
        r'^Computer Engineering$',
        r'^Computer Science$',
        r'^Software Development$',
        r'^Software Engineering$',
        r'^Data Science$',
        r'^Data Engineering$',
        r'^Data Analysis$',
        r'^Machine Learning$',
        r'^Deep Learning$',
    ]
    
    cleaned = []
    for skill in skills_list:
        if not skill or pd.isna(skill):
            continue
        
        skill = str(skill).strip()
        
        # Skip if matches any noise pattern
        is_noise = False
        for pattern in noise_patterns:
            if re.match(pattern, skill, re.IGNORECASE):
                is_noise = True
                break
        
        # Skip short skills (likely categories)
        if len(skill) < 3:
            continue
            
        if not is_noise:
            cleaned.append(skill)
    
    return list(set(cleaned))  # Remove duplicates


def clean_location(location_str: str) -> Dict[str, str]:
    """
    Parse location string into city and country
    Examples: "Cairo, Egypt" → city='Cairo', country='Egypt'
              "Riyadh, Saudi Arabia" → city='Riyadh', country='Saudi Arabia'
    """
    if pd.isna(location_str) or not location_str:
        return {'city': 'Unknown', 'country': 'Unknown'}
    
    parts = [p.strip() for p in str(location_str).split(',')]
    
    if len(parts) >= 2:
        city = parts[0].strip()
        country = parts[-1].strip()
    elif len(parts) == 1:
        city = parts[0].strip()
        country = 'Egypt'  # Default to Egypt
    else:
        city = 'Unknown'
        country = 'Unknown'
    
    return {'city': city, 'country': country}


def clean_company(company_str: str) -> str:
    """Clean company name, replace placeholders with Unknown"""
    if pd.isna(company_str) or not company_str:
        return 'Unknown'
    
    company = str(company_str).strip()
    
    # Replace placeholders
    if company.lower() in ['experienced', 'confidential', '']:
        return 'Unknown'
    
    return company


# Test the functions
print("Testing cleaning functions...")
test_exp = "· 2 - 5 Yrs of Exp"
print(f"Experience: '{test_exp}' → {clean_experience(test_exp)}")

test_skills = ['Vertex Technologies -', 'Experienced', '· Python', 'Data Engineering', 'SQL']
print(f"Skills: {test_skills} → {clean_skills(test_skills)}")

test_loc = "Maadi, Cairo, Egypt"
print(f"Location: '{test_loc}' → {clean_location(test_loc)}")

test_company = "Experienced"
print(f"Company: '{test_company}' → '{clean_company(test_company)}'")

Testing cleaning functions...
Experience: '· 2 - 5 Yrs of Exp' → {'min_experience': 2, 'max_experience': 5, 'experience_level': 'mid'}
Skills: ['Vertex Technologies -', 'Experienced', '· Python', 'Data Engineering', 'SQL'] → ['SQL']
Location: 'Maadi, Cairo, Egypt' → {'city': 'Maadi', 'country': 'Egypt'}
Company: 'Experienced' → 'Unknown'


In [17]:
# ============================================================
# STEP 4: Apply Transformations to Dataset
# ============================================================

# Create a new cleaned DataFrame
cleaned_df = combined_df.copy()

# 1. Clean company names
print("🧹 Cleaning company names...")
cleaned_df['company_name'] = cleaned_df['company'].apply(clean_company)

# 2. Parse experience fields
print("🧹 Parsing experience fields...")
experience_parsed = cleaned_df['experience'].apply(clean_experience)
cleaned_df['min_experience'] = experience_parsed.apply(lambda x: x['min_experience'])
cleaned_df['max_experience'] = experience_parsed.apply(lambda x: x['max_experience'])
cleaned_df['experience_level'] = experience_parsed.apply(lambda x: x['experience_level'])

# 3. Parse location fields
print("🧹 Parsing locations...")
location_parsed = cleaned_df['location'].apply(clean_location)
cleaned_df['city'] = location_parsed.apply(lambda x: x['city'])
cleaned_df['country'] = location_parsed.apply(lambda x: x['country'])

# 4. Clean skills
print("🧹 Cleaning skills...")
cleaned_df['skills_list'] = cleaned_df['skills'].apply(clean_skills)

# 5. Rename and select final columns
print("🧹 Selecting final columns...")
final_df = cleaned_df[[
    'job_id',
    'title',
    'company_name',
    'location',
    'city',
    'country',
    'job_type',
    'experience',
    'min_experience',
    'max_experience',
    'experience_level',
    'skills_list',
    'url',
    'keyword',
    'posted_date',
    'scraped_at'
]].copy()

# Rename columns for clarity
final_df.columns = [
    'job_id',
    'job_title',
    'company_name',
    'location_raw',
    'city',
    'country',
    'job_type',
    'experience_raw',
    'min_experience',
    'max_experience',
    'experience_level',
    'skills_list',
    'job_url',
    'job_category',
    'posted_date',
    'scraped_at'
]

print(f"\n✅ Cleaned dataset shape: {final_df.shape}")
print(f"\n📊 Columns: {final_df.columns.tolist()}")

🧹 Cleaning company names...
🧹 Parsing experience fields...
🧹 Parsing locations...
🧹 Cleaning skills...
🧹 Selecting final columns...

✅ Cleaned dataset shape: (67, 16)

📊 Columns: ['job_id', 'job_title', 'company_name', 'location_raw', 'city', 'country', 'job_type', 'experience_raw', 'min_experience', 'max_experience', 'experience_level', 'skills_list', 'job_url', 'job_category', 'posted_date', 'scraped_at']


In [18]:
# ============================================================
# STEP 5: Verify Cleaned Data Quality
# ============================================================

print("=" * 60)
print("CLEANED DATA VERIFICATION")
print("=" * 60)

print("\n📊 Experience Level Distribution:")
print(final_df['experience_level'].value_counts(dropna=False))

print("\n📊 City Distribution (Top 10):")
print(final_df['city'].value_counts().head(10))

print("\n📊 Country Distribution:")
print(final_df['country'].value_counts())

print("\n📊 Job Category Distribution:")
print(final_df['job_category'].value_counts())

print("\n📊 Job Type Distribution:")
print(final_df['job_type'].value_counts())

print("\n📊 Sample Cleaned Records:")
final_df[['job_title', 'company_name', 'city', 'experience_level', 'skills_list']].head(5)

CLEANED DATA VERIFICATION

📊 Experience Level Distribution:
experience_level
mid          33
senior       22
junior        8
executive     3
NaN           1
Name: count, dtype: int64

📊 City Distribution (Top 10):
city
Cairo          16
Maadi           7
Nasr City       6
New Cairo       5
Sheraton        4
Heliopolis      4
Giza            3
Obour City      3
Riyadh          3
New Capital     2
Name: count, dtype: int64

📊 Country Distribution:
country
Egypt            60
Saudi Arabia      3
Libya             1
United States     1
Germany           1
France            1
Name: count, dtype: int64

📊 Job Category Distribution:
job_category
data engineer        21
backend developer    16
data analyst         15
machine learning     15
Name: count, dtype: int64

📊 Job Type Distribution:
job_type
Full Time              62
Freelance / Project     3
Part Time               2
Name: count, dtype: int64

📊 Sample Cleaned Records:


,job_title,company_name,city,experience_level,skills_list
0,Middle Data Engineer,Vertex Technologies,Cairo,senior,"[Databricks, Relational Databases]"
1,Senior Data Engineer,Vertex Technologies,Cairo,senior,[Databricks]
2,Data Engineer,Hyundai Rotem,Cairo,junior,[Data Modeling]
3,"Data Center Engineer (Compute, Storage & Data ...",Unknown,New Capital,senior,[Data Center]
4,Infrastructure Network Operations Engineer (Da...,Unknown,New Capital,executive,[Data Center Networking]


In [19]:
# ============================================================
# STEP 6: Analyze Top Skills
# ============================================================

from collections import Counter

# Flatten all skills
all_skills = []
for skills in final_df['skills_list']:
    if isinstance(skills, list):
        all_skills.extend(skills)

# Count skills
skill_counts = Counter(all_skills)
top_skills = skill_counts.most_common(20)

print("=" * 60)
print("TOP 20 MOST DEMANDED SKILLS")
print("=" * 60)
for skill, count in top_skills:
    print(f"  {skill:30} → {count:3} jobs")

# Check for duplicates
print("\n" + "=" * 60)
print("DUPLICATE CHECK")
print("=" * 60)
duplicates = final_df[final_df.duplicated(subset=['job_url'], keep=False)]
print(f"Duplicate job URLs: {len(duplicates)}")

# Check missing values
print("\n" + "=" * 60)
print("MISSING VALUES SUMMARY")
print("=" * 60)
missing = final_df.isnull().sum()
print(missing[missing > 0])

TOP 20 MOST DEMANDED SKILLS
  Electrical Engineering         →   3 jobs
  Databricks                     →   2 jobs
  Civil Engineering              →   2 jobs
  Mechanical Engineering         →   2 jobs
  Data Analytics                 →   2 jobs
  Relational Databases           →   1 jobs
  Data Modeling                  →   1 jobs
  Data Center                    →   1 jobs
  Data Center Networking         →   1 jobs
  Traffic Engineering            →   1 jobs
  Data Centers & Mission         →   1 jobs
  Automation Engineering         →   1 jobs
  Manufacturing Engineering      →   1 jobs
  BI Engineer - SQL / PYTHON     →   1 jobs
  Data visualization             →   1 jobs
  Database SQL Server            →   1 jobs
  Big Data Analytics             →   1 jobs
  Financial Data Analysis        →   1 jobs
  Process Analyst                →   1 jobs
  Data Mapping                   →   1 jobs

DUPLICATE CHECK
Duplicate job URLs: 4

MISSING VALUES SUMMARY
min_experience       1
max_ex

In [20]:
# ============================================================
# STEP 7: Refine Skills Cleaning (Keep more technical skills)
# ============================================================

def clean_skills_v2(skills_list: List[str]) -> List[str]:
    """
    Improved skills cleaning - keep more technical skills
    """
    if skills_list is None or len(skills_list) == 0:
        return []
    
    # Items to always remove
    remove_items = {
        'Experienced', 'Entry Level', 'Manager', 'Confidential',
        'IT/Software Development', 'Analyst/Research', 'Engineering',
        'Computer Engineering', 'Computer Science', 'Software Development',
        'Software Engineering', 'Data Science', 'Data Engineering',
        'Data Analysis', 'Machine Learning', 'Deep Learning',
        'Business Analysis', 'Sales/Retail', 'Accounting/Finance',
        'Logistics/Supply Chain', 'Creative/Design/Art', 'R&D/Science',
        'Engineering - Other', 'Purchasing/Procurement', 'Writing/Editorial',
        'B2B Sales', 'Finance', 'Commerce', 'Pharmaceutical'
    }
    
    cleaned = []
    for skill in skills_list:
        if not skill or pd.isna(skill):
            continue
        
        skill = str(skill).strip()
        
        # Skip if it's in the remove list
        if skill in remove_items:
            continue
            
        # Skip if starts with bullet
        if skill.startswith('·'):
            continue
            
        # Skip if ends with dash (company name)
        if skill.endswith('-'):
            continue
            
        # Skip very short items
        if len(skill) < 2:
            continue
        
        cleaned.append(skill)
    
    return list(set(cleaned))


# Re-apply with improved function
print("🧹 Re-cleaning skills with improved function...")
final_df['skills_list'] = combined_df['skills'].apply(clean_skills_v2)

# Re-analyze skills
all_skills = []
for skills in final_df['skills_list']:
    if isinstance(skills, list):
        all_skills.extend(skills)

skill_counts = Counter(all_skills)
top_skills = skill_counts.most_common(25)

print("\n" + "=" * 60)
print("TOP 25 MOST DEMANDED SKILLS (IMPROVED)")
print("=" * 60)
for skill, count in top_skills:
    print(f"  {skill:35} → {count:3} jobs")

🧹 Re-cleaning skills with improved function...

TOP 25 MOST DEMANDED SKILLS (IMPROVED)
  Electrical Engineering              →   3 jobs
  Databricks                          →   2 jobs
  Civil Engineering                   →   2 jobs
  Mechanical Engineering              →   2 jobs
  Data Analytics                      →   2 jobs
  Relational Databases                →   1 jobs
  Data Modeling                       →   1 jobs
  Data Center                         →   1 jobs
  Data Center Networking              →   1 jobs
  Traffic Engineering                 →   1 jobs
  Data Centers & Mission              →   1 jobs
  Automation Engineering              →   1 jobs
  Manufacturing Engineering           →   1 jobs
  BI Engineer - SQL / PYTHON          →   1 jobs
  Data visualization                  →   1 jobs
  Database SQL Server                 →   1 jobs
  Big Data Analytics                  →   1 jobs
  Financial Data Analysis             →   1 jobs
  Process Analyst              

In [21]:
# Debug: Check raw skills data
print("📋 Sample raw skills from original data:")
for i in range(5):
    print(f"\nJob {i}: {combined_df['skills'].iloc[i]}")

# Let's see all unique skills in the raw data
print("\n\n📊 All unique skill values (first 50):")
all_raw_skills = set()
for skills in combined_df['skills']:
    if isinstance(skills, list):
        all_raw_skills.update(skills)

print(sorted(list(all_raw_skills))[:50])

📋 Sample raw skills from original data:

Job 0: ['Vertex Technologies -', 'Experienced', '· IT/Software Development', 'Data Engineering', '· Python', '· Scala', 'Databricks', '· Snowflake', '· SQL', 'Relational Databases']

Job 1: ['Vertex Technologies -', 'Experienced', '· IT/Software Development', 'Data Engineering', '· Python', '· Scala', 'Databricks', '· SQL', '· Dimensional Modeling', '· OLTP/OLAP']

Job 2: ['Hyundai Rotem -', 'Entry Level', '· Analyst/Research', 'Engineering', '· Computer Science', '· English', '· Communication skills', '· ICDL', 'Computer Engineering', 'Data Engineering', 'Data Modeling']

Job 3: ['Experienced', '· IT/Software Development', 'Data Center', '· Cisco UCS', '· Nutanix', '· Storage', '· Backup', '· Veeam', '· Pure Storage', '· HCI']

Job 4: ['Experienced', '· IT/Software Development', '· Cisco ACI', 'Data Center Networking', '· WAN', '· MPLS', '· SD-WAN', '· BGP', '· OSPF', '· Routing & Switching']


📊 All unique skill values (first 50):
['APEX Pharm

In [22]:
# Check all unique skills
print("📊 All unique skill values (50-100):")
print(sorted(list(all_raw_skills))[50:100])

print("\n📊 All unique skill values (100-150):")
print(sorted(list(all_raw_skills))[100:150])

📊 All unique skill values (50-100):
['Makkook.AI -', 'Manager', 'Manufacturing Engineering', 'Mechanical Engineering', 'Mobi Egypt -', 'NX CREATIVE -', 'Ossouss Global Commerce -', 'Prismatecs -', 'Process Analyst', 'Raya Distribution -', 'Relational Databases', 'Sana Commerce -', 'Seero -', 'Software Development', 'Software Engineering', 'Software House Solution -', 'Traffic Engineering', 'Vertex Technologies -', 'Zinox&Master -', 'Zworld Holding -', 'iLock -', 'willys kitchen-Egypt -', '· .Net', '· AI', '· AI Architecture', '· AI Tools & Frameworks', '· API', '· ASP.NET Core', '· ASP.NET Core APIs', '· AWS', '· Accounting', '· Accounting/Finance', '· Advanced Excel', '· Agentic', '· Agentic frameworks', '· Agile', '· Ai', '· Algorithms', '· Analysis', '· Analyst/Research', '· Analytical skills', '· Angular', '· Artificial Intelligence', '· AutoCAD', '· Automation Engineering', '· B2B Sales', '· B2B sales pipeline', '· BGP', '· BI', '· Backup']

📊 All unique skill values (100-150):
['

In [23]:
# ============================================================
# STEP 8: Final Improved Skills Cleaning
# ============================================================

def clean_skills_final(skills_list: List[str]) -> List[str]:
    """
    Final improved skills cleaning - properly handle bullet prefix and keep technical skills
    """
    if skills_list is None or len(skills_list) == 0:
        return []
    
    # Items to always remove (categories, levels, company placeholders)
    remove_items = {
        'Experienced', 'Entry Level', 'Manager', 'Confidential',
        'IT/Software Development', 'Analyst/Research', 'Engineering',
        'Computer Engineering', 'Computer Science', 'Software Development',
        'Software Engineering', 'Data Science', 'Data Engineering',
        'Data Analysis', 'Machine Learning', 'Deep Learning',
        'Business Analysis', 'Sales/Retail', 'Accounting/Finance',
        'Logistics/Supply Chain', 'Creative/Design/Art', 'R&D/Science',
        'Engineering - Other', 'Purchasing/Procurement', 'Writing/Editorial',
        'B2B Sales', 'Finance', 'Commerce', 'Pharmaceutical'
    }
    
    cleaned = []
    for skill in skills_list:
        if not skill or pd.isna(skill):
            continue
        
        skill = str(skill).strip()
        
        # Remove bullet prefix
        if skill.startswith('· '):
            skill = skill[2:]
        elif skill.startswith('·'):
            skill = skill[1:].strip()
        
        # Skip if it's in the remove list
        if skill in remove_items:
            continue
            
        # Skip if ends with dash (company name)
        if skill.endswith('-'):
            continue
            
        # Skip very short items
        if len(skill) < 2:
            continue
        
        cleaned.append(skill)
    
    return list(set(cleaned))


# Re-apply with final function
print("🧹 Re-cleaning skills with final function...")
final_df['skills_list'] = combined_df['skills'].apply(clean_skills_final)

# Re-analyze skills
all_skills = []
for skills in final_df['skills_list']:
    if isinstance(skills, list):
        all_skills.extend(skills)

skill_counts = Counter(all_skills)
top_skills = skill_counts.most_common(30)

print("\n" + "=" * 60)
print("TOP 30 MOST DEMANDED SKILLS (FINAL)")
print("=" * 60)
for skill, count in top_skills:
    print(f"  {skill:35} → {count:3} jobs")

🧹 Re-cleaning skills with final function...

TOP 30 MOST DEMANDED SKILLS (FINAL)
  Python                              →  14 jobs
  SQL                                 →   8 jobs
  Microsoft Excel                     →   8 jobs
  AI                                  →   6 jobs
  Analysis                            →   6 jobs
  Problem Solving                     →   5 jobs
  Power BI                            →   5 jobs
  Git                                 →   5 jobs
  Financial Analysis                  →   4 jobs
  MySQL                               →   4 jobs
  .Net                                →   4 jobs
  Statistical Analysis                →   3 jobs
  Python Programming                  →   3 jobs
  Communication                       →   3 jobs
  Mechanical Engineering              →   3 jobs
  Electrical Engineering              →   3 jobs
  Project Management                  →   3 jobs
  Budgeting                           →   3 jobs
  Forecasting                        

In [24]:
# ============================================================
# STEP 9: Handle Duplicates and Finalize Dataset
# ============================================================

print("=" * 60)
print("HANDLING DUPLICATES")
print("=" * 60)

# Check for duplicates based on job_url
print(f"\nBefore deduplication: {len(final_df)} jobs")
duplicates = final_df[final_df.duplicated(subset=['job_url'], keep=False)]
print(f"Duplicate records: {len(duplicates)}")

# Show duplicate records
if len(duplicates) > 0:
    print("\nDuplicate job URLs:")
    print(duplicates[['job_title', 'job_url', 'job_category']])

# Remove duplicates
final_df_clean = final_df.drop_duplicates(subset=['job_url'], keep='first')
print(f"\nAfter deduplication: {len(final_df_clean)} jobs")

# Final schema check
print("\n" + "=" * 60)
print("FINAL DATASET SCHEMA")
print("=" * 60)
print(final_df_clean.dtypes)

HANDLING DUPLICATES

Before deduplication: 67 jobs
Duplicate records: 4

Duplicate job URLs:
                                            job_title  \
6   Data Scientist And Machine learning Engineer -...   
12                                        AI Engineer   
52  Data Scientist And Machine learning Engineer -...   
54                                        AI Engineer   

                                              job_url      job_category  
6   https://wuzzuf.net/jobs/p/hif824t4axxr-data-sc...     data engineer  
12  https://wuzzuf.net/jobs/p/0jlu7uzk7kis-ai-engi...     data engineer  
52  https://wuzzuf.net/jobs/p/hif824t4axxr-data-sc...  machine learning  
54  https://wuzzuf.net/jobs/p/0jlu7uzk7kis-ai-engi...  machine learning  

After deduplication: 65 jobs

FINAL DATASET SCHEMA
job_id                  str
job_title               str
company_name            str
location_raw            str
city                    str
country                 str
job_type                str
exp

In [25]:
# ============================================================
# STEP 10: Final Column Selection and Export
# ============================================================

# Select final columns (keep core analytics columns)
final_columns = [
    'job_title',
    'company_name',
    'location_raw',
    'city',
    'country',
    'job_type',
    'experience_raw',
    'min_experience',
    'max_experience',
    'experience_level',
    'skills_list',
    'job_category',
    'posted_date',
    'job_url'
]

final_df_final = final_df_clean[final_columns].copy()

# Reset index
final_df_final = final_df_final.reset_index(drop=True)

print("=" * 60)
print("FINAL CLEANED DATASET")
print("=" * 60)
print(f"\nShape: {final_df_final.shape}")
print(f"\nColumns: {final_df_final.columns.tolist()}")

print("\n📊 Final Data Summary:")
print(f"  • Total Jobs: {len(final_df_final)}")
print(f"  • Unique Companies: {final_df_final['company_name'].nunique()}")
print(f"  • Unique Cities: {final_df_final['city'].nunique()}")
print(f"  • Job Categories: {final_df_final['job_category'].nunique()}")

print("\n📊 Experience Level Distribution:")
print(final_df_final['experience_level'].value_counts())

print("\n📊 Sample of Final Data:")
final_df_final.head(3)

FINAL CLEANED DATASET

Shape: (65, 14)

Columns: ['job_title', 'company_name', 'location_raw', 'city', 'country', 'job_type', 'experience_raw', 'min_experience', 'max_experience', 'experience_level', 'skills_list', 'job_category', 'posted_date', 'job_url']

📊 Final Data Summary:
  • Total Jobs: 65
  • Unique Companies: 47
  • Unique Cities: 23
  • Job Categories: 4

📊 Experience Level Distribution:
experience_level
mid          32
senior       22
junior        7
executive     3
Name: count, dtype: int64

📊 Sample of Final Data:


,job_title,company_name,location_raw,city,country,job_type,experience_raw,min_experience,max_experience,experience_level,skills_list,job_category,posted_date,job_url
0,Middle Data Engineer,Vertex Technologies,"Cairo, Egypt",Cairo,Egypt,Full Time,· 2+ Yrs of Exp,2.0,NaN,senior,"[Databricks, Snowflake, Python, Scala, Relatio...",data engineer,Middle Data Engineer\nVertex Technologies - Ca...,https://wuzzuf.net/jobs/p/f3fi6wjnx9o9-middle-...
1,Senior Data Engineer,Vertex Technologies,"Cairo, Egypt",Cairo,Egypt,Full Time,· 4+ Yrs of Exp,4.0,NaN,senior,"[Databricks, Dimensional Modeling, Python, Sca...",data engineer,Senior Data Engineer\nVertex Technologies - Ca...,https://wuzzuf.net/jobs/p/dw9vbculsh2b-senior-...
2,Data Engineer,Hyundai Rotem,"Cairo, Egypt",Cairo,Egypt,Full Time,· 1 - 3 Yrs of Exp,1.0,3.0,junior,"[English, ICDL, Communication skills, Data Mod...",data engineer,"Data Engineer\nHyundai Rotem - Cairo, Egypt\n1...",https://wuzzuf.net/jobs/p/beqgj9o9kpfk-data-en...


In [26]:
# ============================================================
# STEP 11: Save Cleaned Dataset
# ============================================================

import os

# Create output directory if it doesn't exist
output_dir = "../data/cleaned"
os.makedirs(output_dir, exist_ok=True)

# Save to CSV
output_path = os.path.join(output_dir, "jobs_cleaned.csv")
final_df_final.to_csv(output_path, index=False, encoding='utf-8')

print(f"✅ Cleaned dataset saved to: {output_path}")
print(f"   Total records: {len(final_df_final)}")

# Also save a JSON version for flexibility
import json
json_path = os.path.join(output_dir, "jobs_cleaned.json")

# Convert skills_list to string for JSON serialization
json_df = final_df_final.copy()
json_df['skills_list'] = json_df['skills_list'].apply(lambda x: json.dumps(x) if isinstance(x, list) else x)

json_df.to_json(json_path, orient='records', indent=2, force_ascii=False)
print(f"✅ JSON version saved to: {json_path}")

# Verify files
print("\n📁 Output files:")
for f in os.listdir(output_dir):
    file_path = os.path.join(output_dir, f)
    size = os.path.getsize(file_path)
    print(f"   • {f} ({size:,} bytes)")

✅ Cleaned dataset saved to: ../data/cleaned\jobs_cleaned.csv
   Total records: 65
✅ JSON version saved to: ../data/cleaned\jobs_cleaned.json

📁 Output files:
   • jobs_cleaned.csv (25,913 bytes)
   • jobs_cleaned.json (45,656 bytes)


In [27]:
# ============================================================
# FINAL SUMMARY: Egypt Job Market Analytics Dataset
# ============================================================

print("=" * 70)
print("📊 EGYPT JOB MARKET ANALYTICS - CLEANED DATASET SUMMARY")
print("=" * 70)

print("\n🎯 DATASET OVERVIEW:")
print(f"   • Total Jobs: {len(final_df_final)}")
print(f"   • Unique Companies: {final_df_final['company_name'].nunique()}")
print(f"   • Unique Cities: {final_df_final['city'].nunique()}")
print(f"   • Job Categories: {final_df_final['job_category'].nunique()}")

print("\n📈 JOB CATEGORY BREAKDOWN:")
for cat, count in final_df_final['job_category'].value_counts().items():
    print(f"   • {cat}: {count} jobs ({count/len(final_df_final)*100:.1f}%)")

print("\n🏢 TOP 5 HIRING COMPANIES:")
for company, count in final_df_final['company_name'].value_counts().head(5).items():
    print(f"   • {company}: {count} jobs")

print("\n🌍 TOP 5 CITIES:")
for city, count in final_df_final['city'].value_counts().head(5).items():
    print(f"   • {city}: {count} jobs")

print("\n💼 EXPERIENCE LEVEL DISTRIBUTION:")
for level, count in final_df_final['experience_level'].value_counts().items():
    print(f"   • {level}: {count} jobs ({count/len(final_df_final)*100:.1f}%)")

print("\n🔝 TOP 10 MOST DEMANDED SKILLS:")
for skill, count in skill_counts.most_common(10):
    print(f"   • {skill}: {count} jobs")

print("\n📁 OUTPUT FILES:")
print(f"   • CSV: data/cleaned/jobs_cleaned.csv")
print(f"   • JSON: data/cleaned/jobs_cleaned.json")

print("\n" + "=" * 70)
print("✅ DATA CLEANING COMPLETE - READY FOR BI & ANALYTICS")
print("=" * 70)

📊 EGYPT JOB MARKET ANALYTICS - CLEANED DATASET SUMMARY

🎯 DATASET OVERVIEW:
   • Total Jobs: 65
   • Unique Companies: 47
   • Unique Cities: 23
   • Job Categories: 4

📈 JOB CATEGORY BREAKDOWN:
   • data engineer: 21 jobs (32.3%)
   • backend developer: 16 jobs (24.6%)
   • data analyst: 15 jobs (23.1%)
   • machine learning: 13 jobs (20.0%)

🏢 TOP 5 HIRING COMPANIES:
   • Unknown: 9 jobs
   • Vertex Technologies: 4 jobs
   • Mobi Egypt: 2 jobs
   • EpsilonAI: 2 jobs
   • Anzemah: 2 jobs

🌍 TOP 5 CITIES:
   • Cairo: 16 jobs
   • Maadi: 7 jobs
   • Nasr City: 5 jobs
   • New Cairo: 5 jobs
   • Heliopolis: 4 jobs

💼 EXPERIENCE LEVEL DISTRIBUTION:
   • mid: 32 jobs (49.2%)
   • senior: 22 jobs (33.8%)
   • junior: 7 jobs (10.8%)
   • executive: 3 jobs (4.6%)

🔝 TOP 10 MOST DEMANDED SKILLS:
   • Python: 14 jobs
   • SQL: 8 jobs
   • Microsoft Excel: 8 jobs
   • AI: 6 jobs
   • Analysis: 6 jobs
   • Problem Solving: 5 jobs
   • Power BI: 5 jobs
   • Git: 5 jobs
   • Financial Analysis: 4 j

In [28]:
# ============================================================
# STEP 12: Load and Explore CSV Data
# ============================================================

# Load the CSV file
csv_path = "../data/raw/wuzzuf_jobs_raw.csv"
df_csv = pd.read_csv(csv_path)

print("=" * 60)
print("CSV DATA EXPLORATION")
print("=" * 60)

print(f"\nShape: {df_csv.shape}")
print(f"\nColumns: {df_csv.columns.tolist()}")

print("\n📋 Sample record:")
print(df_csv.iloc[0])

print("\n📊 Data types:")
print(df_csv.dtypes)

CSV DATA EXPLORATION

Shape: (631, 14)

Columns: ['job_url', 'source_platform', 'search_keywords', 'scraped_at', 'job_title', 'company_location_text', 'posted_date', 'job_type', 'workplace_type', 'experience', 'salary', 'skills', 'job_description', 'job_requirements']

📋 Sample record:
job_url                  https://wuzzuf.net/jobs/p/6357bgijgsem-sales-m...
source_platform                                                     Wuzzuf
search_keywords          ['data analyst', 'business analyst', 'power bi...
scraped_at                                             2026-04-25 03:31:17
job_title                                   Sales & Marketing Data Analyst
company_location_text                                                  NaN
posted_date                                              posted 2 days ago
job_type                                                         Full Time
workplace_type                                                     On-site
experience                            

In [29]:
# Explore key fields in CSV
print("📊 Experience field samples:")
print(df_csv['experience'].value_counts().head(10))

print("\n📊 Skills field samples:")
print(df_csv['skills'].iloc[0])

print("\n📊 Job Type distribution:")
print(df_csv['job_type'].value_counts())

print("\n📊 Workplace Type distribution:")
print(df_csv['workplace_type'].value_counts())

print("\n📊 Sample job titles:")
print(df_csv['job_title'].head(10).tolist())

print("\n📊 Sample search_keywords:")
print(df_csv['search_keywords'].iloc[0])

📊 Experience field samples:
experience
3 To 7 Years         91
3 To 5 Years         64
1 To 3 Years         47
2 To 4 Years         32
2 To 5 Years         32
5 To 7 Years         29
0 To 2 Years         27
5 To 10 Years        27
1 To 2 Years         25
More Than 3 Years    17
Name: count, dtype: int64

📊 Skills field samples:
['Data Analysis', 'Market Research', 'Marketing', 'Sales']

📊 Job Type distribution:
job_type
Full Time              603
Internship              11
Freelance / Project     10
Part Time                4
دوام كامل                2
Shift Based              1
Name: count, dtype: int64

📊 Workplace Type distribution:
workplace_type
On-site                516
Hybrid                  65
Remote                  38
Part Time                6
Freelance / Project      3
عمل من مقر الشركة        2
Shift Based              1
Name: count, dtype: int64

📊 Sample job titles:
['Sales & Marketing Data Analyst', 'Data Analyst', 'Data Analyst', 'Data Analyst', 'Supply Chain data An

In [30]:
# ============================================================
# STEP 13: Clean CSV Data with Functions Matching JSON Schema
# ============================================================
import ast

def clean_experience_csv(exp_str: str) -> Dict[str, Any]:
    """
    Parse CSV experience string
    Examples: "0 To 2 Years" → min=0, max=2, level='junior'
              "3 To 7 Years" → min=3, max=7, level='mid'
              "More Than 3 Years" → min=3, max=None, level='senior'
    """
    if pd.isna(exp_str) or not exp_str:
        return {'min_experience': None, 'max_experience': None, 'experience_level': None}
    
    exp_clean = str(exp_str).strip()
    
    # Pattern 1: "X To Y Years"
    range_match = re.search(r'(\d+)\s*To\s*(\d+)\s*Years', exp_clean)
    if range_match:
        min_exp = int(range_match.group(1))
        max_exp = int(range_match.group(2))
        level = categorize_experience(min_exp, max_exp)
        return {'min_experience': min_exp, 'max_experience': max_exp, 'experience_level': level}
    
    # Pattern 2: "More Than X Years"
    more_match = re.search(r'More Than (\d+) Years', exp_clean)
    if more_match:
        min_exp = int(more_match.group(1))
        return {'min_experience': min_exp, 'max_experience': None, 'experience_level': 'senior'}
    
    # Pattern 3: "X Years" (single number)
    single_match = re.search(r'(\d+)\s*Years', exp_clean)
    if single_match:
        exp = int(single_match.group(1))
        return {'min_experience': exp, 'max_experience': exp, 'experience_level': categorize_experience(exp, exp)}
    
    return {'min_experience': None, 'max_experience': None, 'experience_level': None}


def clean_skills_csv(skills_str: str) -> List[str]:
    """
    Parse CSV skills string (stored as string representation of list)
    """
    if pd.isna(skills_str) or not skills_str:
        return []
    
    try:
        # Try to parse as Python list literal
        skills_list = ast.literal_eval(skills_str)
        if isinstance(skills_list, list):
            return clean_skills_final(skills_list)
    except:
        pass
    
    # If parsing fails, return as-is
    return []


def extract_job_category(keywords_str: str) -> str:
    """
    Extract primary job category from search_keywords
    """
    if pd.isna(keywords_str) or not keywords_str:
        return 'unknown'
    
    try:
        keywords = ast.literal_eval(keywords_str)
        if isinstance(keywords, list) and len(keywords) > 0:
            return keywords[0]
    except:
        pass
    
    return 'unknown'


def clean_job_type_csv(job_type: str) -> str:
    """Normalize job type"""
    if pd.isna(job_type):
        return 'Unknown'
    
    job_type = str(job_type).strip()
    
    # Map variations
    type_mapping = {
        'Full Time': 'Full Time',
        'دوام كامل': 'Full Time',
        'Part Time': 'Part Time',
        'Internship': 'Internship',
        'Freelance / Project': 'Freelance / Project',
        'Shift Based': 'Shift Based'
    }
    
    return type_mapping.get(job_type, job_type)


def clean_workplace_type(workplace: str) -> str:
    """Normalize workplace type"""
    if pd.isna(workplace):
        return 'On-site'
    
    workplace = str(workplace).strip()
    
    type_mapping = {
        'On-site': 'On-site',
        'عمل من مقر الشركة': 'On-site',
        'Hybrid': 'Hybrid',
        'Remote': 'Remote',
        'Part Time': 'Part Time',
        'Freelance / Project': 'Freelance / Project',
        'Shift Based': 'Shift Based'
    }
    
    return type_mapping.get(workplace, workplace)


# Test CSV cleaning functions
print("Testing CSV cleaning functions...")
test_exp_csv = "0 To 2 Years"
print(f"Experience: '{test_exp_csv}' → {clean_experience_csv(test_exp_csv)}")

test_exp_csv2 = "More Than 3 Years"
print(f"Experience: '{test_exp_csv2}' → {clean_experience_csv(test_exp_csv2)}")

test_skills_csv = "['Data Analysis', 'Market Research', 'Marketing', 'Sales']"
print(f"Skills: '{test_skills_csv}' → {clean_skills_csv(test_skills_csv)}")

test_keywords = "['data analyst', 'business analyst', 'power bi']"
print(f"Keywords: '{test_keywords}' → {extract_job_category(test_keywords)}")

Testing CSV cleaning functions...
Experience: '0 To 2 Years' → {'min_experience': 0, 'max_experience': 2, 'experience_level': 'junior'}
Experience: 'More Than 3 Years' → {'min_experience': 3, 'max_experience': None, 'experience_level': 'senior'}
Skills: '['Data Analysis', 'Market Research', 'Marketing', 'Sales']' → ['Sales', 'Market Research', 'Marketing']
Keywords: '['data analyst', 'business analyst', 'power bi']' → data analyst


In [31]:
# ============================================================
# STEP 14: Apply Transformations to CSV Data
# ============================================================

# Create a copy for transformation
csv_cleaned = df_csv.copy()

# 1. Parse experience fields
print("🧹 Parsing experience fields...")
experience_parsed = csv_cleaned['experience'].apply(clean_experience_csv)
csv_cleaned['min_experience'] = experience_parsed.apply(lambda x: x['min_experience'])
csv_cleaned['max_experience'] = experience_parsed.apply(lambda x: x['max_experience'])
csv_cleaned['experience_level'] = experience_parsed.apply(lambda x: x['experience_level'])

# 2. Clean skills
print("🧹 Cleaning skills...")
csv_cleaned['skills_list'] = csv_cleaned['skills'].apply(clean_skills_csv)

# 3. Extract job category from search_keywords
print("🧹 Extracting job categories...")
csv_cleaned['job_category'] = csv_cleaned['search_keywords'].apply(extract_job_category)

# 4. Normalize job type
print("🧹 Normalizing job types...")
csv_cleaned['job_type_clean'] = csv_cleaned['job_type'].apply(clean_job_type_csv)

# 5. Normalize workplace type
print("🧹 Normalizing workplace types...")
csv_cleaned['workplace_type_clean'] = csv_cleaned['workplace_type'].apply(clean_workplace_type)

# 6. Set company name (not available in CSV, use Unknown)
csv_cleaned['company_name'] = 'Unknown'

# 7. Set location (not available in CSV, use Unknown)
csv_cleaned['location_raw'] = 'Unknown'
csv_cleaned['city'] = 'Unknown'
csv_cleaned['country'] = 'Unknown'

print(f"\n✅ CSV transformed: {len(csv_cleaned)} records")

🧹 Parsing experience fields...
🧹 Cleaning skills...
🧹 Extracting job categories...
🧹 Normalizing job types...
🧹 Normalizing workplace types...

✅ CSV transformed: 631 records


In [32]:
# ============================================================
# STEP 15: Create Final CSV DataFrame with Matching Schema
# ============================================================

# Select and rename columns to match JSON schema
csv_final = pd.DataFrame({
    'job_title': csv_cleaned['job_title'],
    'company_name': csv_cleaned['company_name'],
    'location_raw': csv_cleaned['location_raw'],
    'city': csv_cleaned['city'],
    'country': csv_cleaned['country'],
    'job_type': csv_cleaned['job_type_clean'],
    'experience_raw': csv_cleaned['experience'],
    'min_experience': csv_cleaned['min_experience'],
    'max_experience': csv_cleaned['max_experience'],
    'experience_level': csv_cleaned['experience_level'],
    'skills_list': csv_cleaned['skills_list'],
    'job_category': csv_cleaned['job_category'],
    'posted_date': csv_cleaned['posted_date'],
    'job_url': csv_cleaned['job_url']
})

# Add source column to track origin
csv_final['data_source'] = 'csv_wuzzuf_jobs'

print("=" * 60)
print("CSV CLEANED DATASET")
print("=" * 60)
print(f"\nShape: {csv_final.shape}")
print(f"\nColumns: {csv_final.columns.tolist()}")

print("\n📊 Experience Level Distribution:")
print(csv_final['experience_level'].value_counts())

print("\n📊 Job Category Distribution (Top 10):")
print(csv_final['job_category'].value_counts().head(10))

print("\n📊 Sample of CSV Cleaned Data:")
csv_final.head(3)

CSV CLEANED DATASET

Shape: (631, 15)

Columns: ['job_title', 'company_name', 'location_raw', 'city', 'country', 'job_type', 'experience_raw', 'min_experience', 'max_experience', 'experience_level', 'skills_list', 'job_category', 'posted_date', 'job_url', 'data_source']

📊 Experience Level Distribution:
experience_level
mid          285
senior       168
junior       112
executive     38
Name: count, dtype: int64

📊 Job Category Distribution (Top 10):
job_category
data engineer              140
data analyst               103
analytics engineer         103
sql developer               89
ai engineer                 42
power bi                    36
business analyst            25
python developer data       18
data pipeline engineer      17
data warehouse engineer     13
Name: count, dtype: int64

📊 Sample of CSV Cleaned Data:


,job_title,company_name,location_raw,city,country,job_type,experience_raw,min_experience,max_experience,experience_level,skills_list,job_category,posted_date,job_url,data_source
0,Sales & Marketing Data Analyst,Unknown,Unknown,Unknown,Unknown,Full Time,0 To 2 Years,0.0,2.0,junior,"[Sales, Market Research, Marketing]",data analyst,posted 2 days ago,https://wuzzuf.net/jobs/p/6357bgijgsem-sales-m...,csv_wuzzuf_jobs
1,Data Analyst,Unknown,Unknown,Unknown,Unknown,Full Time,2 To 4 Years,2.0,4.0,mid,"[Microsoft Excel, Python, Power BI]",data analyst,posted 4 days ago,https://wuzzuf.net/jobs/p/485eob7kytt8-data-an...,csv_wuzzuf_jobs
2,Data Analyst,Unknown,Unknown,Unknown,Unknown,Full Time,2 To 3 Years,2.0,3.0,mid,"[Python, Power BI, Information Technology (IT)...",data analyst,posted 10 days ago,https://wuzzuf.net/jobs/p/fzv9lnkvfsdn-data-an...,csv_wuzzuf_jobs


In [33]:
# ============================================================
# STEP 16: Combine JSON and CSV Datasets
# ============================================================

# Add data_source to JSON dataset for consistency
final_df_final['data_source'] = 'json_wuzzuf'

# Ensure both datasets have the same columns
json_cols = final_df_final.columns.tolist()
csv_cols = csv_final.columns.tolist()

print("JSON columns:", json_cols)
print("CSV columns:", csv_cols)

# Combine datasets
combined_final = pd.concat([final_df_final, csv_final], ignore_index=True)

print(f"\n✅ Combined dataset: {len(combined_final)} total records")
print(f"   • From JSON: {len(final_df_final)}")
print(f"   • From CSV: {len(csv_final)}")

# Check for duplicates
duplicates = combined_final[combined_final.duplicated(subset=['job_url'], keep=False)]
print(f"\n⚠️  Duplicate job URLs: {len(duplicates)}")

# Remove duplicates
combined_final_clean = combined_final.drop_duplicates(subset=['job_url'], keep='first')
print(f"✅ After deduplication: {len(combined_final_clean)} records")

JSON columns: ['job_title', 'company_name', 'location_raw', 'city', 'country', 'job_type', 'experience_raw', 'min_experience', 'max_experience', 'experience_level', 'skills_list', 'job_category', 'posted_date', 'job_url', 'data_source']
CSV columns: ['job_title', 'company_name', 'location_raw', 'city', 'country', 'job_type', 'experience_raw', 'min_experience', 'max_experience', 'experience_level', 'skills_list', 'job_category', 'posted_date', 'job_url', 'data_source']

✅ Combined dataset: 696 total records
   • From JSON: 65
   • From CSV: 631

⚠️  Duplicate job URLs: 106
✅ After deduplication: 643 records


In [34]:
# ============================================================
# STEP 17: Save Final Combined Dataset
# ============================================================

# Reset index
combined_final_clean = combined_final_clean.reset_index(drop=True)

# Save to CSV
output_path = "../data/cleaned/jobs_combined.csv"
combined_final_clean.to_csv(output_path, index=False, encoding='utf-8')

print(f"✅ Combined dataset saved to: {output_path}")
print(f"   Total records: {len(combined_final_clean)}")

# Also save JSON version
json_output_path = "../data/cleaned/jobs_combined.json"
json_df = combined_final_clean.copy()
json_df['skills_list'] = json_df['skills_list'].apply(lambda x: json.dumps(x) if isinstance(x, list) else x)
json_df.to_json(json_output_path, orient='records', indent=2, force_ascii=False)
print(f"✅ JSON version saved to: {json_output_path}")

# Verify files
print("\n📁 Output files in data/cleaned/:")
for f in os.listdir("../data/cleaned/"):
    file_path = os.path.join("../data/cleaned/", f)
    size = os.path.getsize(file_path)
    print(f"   • {f} ({size:,} bytes)")

✅ Combined dataset saved to: ../data/cleaned/jobs_combined.csv
   Total records: 643
✅ JSON version saved to: ../data/cleaned/jobs_combined.json

📁 Output files in data/cleaned/:
   • jobs_cleaned.csv (25,913 bytes)
   • jobs_cleaned.json (45,656 bytes)
   • jobs_combined.csv (252,773 bytes)
   • jobs_combined.json (468,052 bytes)


In [35]:
# ============================================================
# FINAL SUMMARY: Combined Egypt Job Market Dataset
# ============================================================

print("=" * 70)
print("📊 EGYPT JOB MARKET ANALYTICS - COMBINED DATASET SUMMARY")
print("=" * 70)

print("\n🎯 DATASET OVERVIEW:")
print(f"   • Total Jobs: {len(combined_final_clean)}")
print(f"   • Unique Companies: {combined_final_clean['company_name'].nunique()}")
print(f"   • Unique Cities: {combined_final_clean['city'].nunique()}")
print(f"   • Job Categories: {combined_final_clean['job_category'].nunique()}")

print("\n📈 DATA SOURCE BREAKDOWN:")
for source, count in combined_final_clean['data_source'].value_counts().items():
    print(f"   • {source}: {count} jobs ({count/len(combined_final_clean)*100:.1f}%)")

print("\n📈 JOB CATEGORY BREAKDOWN (Top 15):")
for cat, count in combined_final_clean['job_category'].value_counts().head(15).items():
    print(f"   • {cat}: {count} jobs ({count/len(combined_final_clean)*100:.1f}%)")

print("\n🏢 TOP 10 HIRING COMPANIES:")
for company, count in combined_final_clean['company_name'].value_counts().head(10).items():
    print(f"   • {company}: {count} jobs")

print("\n🌍 TOP 10 CITIES:")
for city, count in combined_final_clean['city'].value_counts().head(10).items():
    print(f"   • {city}: {count} jobs")

print("\n💼 EXPERIENCE LEVEL DISTRIBUTION:")
for level, count in combined_final_clean['experience_level'].value_counts().items():
    print(f"   • {level}: {count} jobs ({count/len(combined_final_clean)*100:.1f}%)")

# Get skills from combined dataset
all_skills_combined = []
for skills in combined_final_clean['skills_list']:
    if isinstance(skills, list):
        all_skills_combined.extend(skills)

skill_counts_combined = Counter(all_skills_combined)
print("\n🔝 TOP 15 MOST DEMANDED SKILLS:")
for skill, count in skill_counts_combined.most_common(15):
    print(f"   • {skill}: {count} jobs")

print("\n📁 FINAL OUTPUT FILES:")
print(f"   • CSV: data/cleaned/jobs_combined.csv")
print(f"   • JSON: data/cleaned/jobs_combined.json")

print("\n📋 SCHEMA:")
print(f"   {combined_final_clean.columns.tolist()}")

print("\n" + "=" * 70)
print("✅ DATA CLEANING COMPLETE - READY FOR DBT & ETL PIPELINES")
print("=" * 70)

📊 EGYPT JOB MARKET ANALYTICS - COMBINED DATASET SUMMARY

🎯 DATASET OVERVIEW:
   • Total Jobs: 643
   • Unique Companies: 47
   • Unique Cities: 24
   • Job Categories: 20

📈 DATA SOURCE BREAKDOWN:
   • csv_wuzzuf_jobs: 578 jobs (89.9%)
   • json_wuzzuf: 65 jobs (10.1%)

📈 JOB CATEGORY BREAKDOWN (Top 15):
   • data engineer: 141 jobs (21.9%)
   • analytics engineer: 103 jobs (16.0%)
   • data analyst: 98 jobs (15.2%)
   • sql developer: 83 jobs (12.9%)
   • ai engineer: 42 jobs (6.5%)
   • power bi: 36 jobs (5.6%)
   • business analyst: 25 jobs (3.9%)
   • backend developer: 16 jobs (2.5%)
   • python developer data: 16 jobs (2.5%)
   • data pipeline engineer: 16 jobs (2.5%)
   • machine learning: 13 jobs (2.0%)
   • data warehouse engineer: 13 jobs (2.0%)
   • tableau developer: 8 jobs (1.2%)
   • machine learning engineer: 8 jobs (1.2%)
   • operations analyst: 6 jobs (0.9%)

🏢 TOP 10 HIRING COMPANIES:
   • Unknown: 587 jobs
   • Vertex Technologies: 4 jobs
   • Mobi Egypt: 2 jobs
   

In [36]:
# Quick summary
print("=" * 70)
print("FINAL COMBINED DATASET - QUICK SUMMARY")
print("=" * 70)

print(f"\n✅ Total Jobs: {len(combined_final_clean)}")
print(f"   • From JSON files: {len(final_df_final)}")
print(f"   • From CSV file: {len(csv_final)}")
print(f"   • After deduplication: {len(combined_final_clean)}")

print(f"\n📊 Data Sources:")
print(combined_final_clean['data_source'].value_counts())

print(f"\n📊 Experience Levels:")
print(combined_final_clean['experience_level'].value_counts())

print(f"\n📊 Top 5 Job Categories:")
print(combined_final_clean['job_category'].value_counts().head())

print(f"\n📁 Files saved:")
print("   • data/cleaned/jobs_combined.csv")
print("   • data/cleaned/jobs_combined.json")

print("\n✅ Ready for DBT & ETL pipelines!")

FINAL COMBINED DATASET - QUICK SUMMARY

✅ Total Jobs: 643
   • From JSON files: 65
   • From CSV file: 631
   • After deduplication: 643

📊 Data Sources:
data_source
csv_wuzzuf_jobs    578
json_wuzzuf         65
Name: count, dtype: int64

📊 Experience Levels:
experience_level
mid          289
senior       175
junior       114
executive     38
Name: count, dtype: int64

📊 Top 5 Job Categories:
job_category
data engineer         141
analytics engineer    103
data analyst           98
sql developer          83
ai engineer            42
Name: count, dtype: int64

📁 Files saved:
   • data/cleaned/jobs_combined.csv
   • data/cleaned/jobs_combined.json

✅ Ready for DBT & ETL pipelines!


In [37]:
# ============================================================
# STEP 18: Extract Location from Job URLs
# ============================================================

def extract_location_from_url(url: str) -> Dict[str, str]:
    """
    Extract location from job URL
    Example: https://wuzzuf.net/jobs/p/zav5d9coyr5v-procurement-analyst-guillam-alexandria-egypt
             → city='Alexandria', country='Egypt'
    """
    if pd.isna(url) or not url:
        return {'city': 'Unknown', 'country': 'Unknown'}
    
    url = str(url).lower()
    
    # Common Egyptian cities
    egyptian_cities = [
        'cairo', 'alexandria', 'giza', 'mansoura', 'tanta', 'zagazig',
        'suez', 'ismailia', 'faiyum', 'beni-suef', 'minya', 'assiut',
        'sohag', 'qena', 'luxor', 'aswan', 'damietta', 'port-said',
        'new-cairo', 'new capital', 'maadi', 'nasr city', 'heliopolis',
        'mohandessin', 'dokki', '6th of october', 'obour city',
        'shorouk city', 'badr city', 'sheraton', 'abbassia'
    ]
    
    # Gulf countries
    saudi_cities = ['riyadh', 'jeddah', 'mecca', 'dammam', 'khobar']
    uae_cities = ['dubai', 'abu dhabi', 'sharjah', 'ajman']
    kuwait_cities = ['kuwait city', 'hawally']
    qatar_cities = ['doha']
    
    # Try to extract location from URL
    # Pattern: ...-city-country
    
    # Check for Egypt
    if '-egypt' in url:
        for city in egyptian_cities:
            if f'-{city}-egypt' in url or f'-{city},-egypt' in url:
                # Clean city name
                city_clean = city.replace('-', ' ').title()
                if city_clean == 'New Cairo':
                    city_clean = 'New Cairo'
                elif city_clean == 'New Capital':
                    city_clean = 'New Capital'
                elif city_clean == '6Th Of October':
                    city_clean = '6th of October'
                elif city_clean == 'Obour City':
                    city_clean = 'Obour City'
                return {'city': city_clean, 'country': 'Egypt'}
        
        # If city not found but Egypt is present
        return {'city': 'Unknown', 'country': 'Egypt'}
    
    # Check for Saudi Arabia
    elif '-saudi-arabia' in url or '-sa' in url:
        for city in saudi_cities:
            if f'-{city}-saudi' in url or f'-{city}-sa' in url:
                return {'city': city.title(), 'country': 'Saudi Arabia'}
        return {'city': 'Unknown', 'country': 'Saudi Arabia'}
    
    # Check for UAE
    elif '-united-arab-emirates' in url or '-uae' in url:
        for city in uae_cities:
            if f'-{city}-uae' in url or f'-{city}-united' in url:
                return {'city': city.title(), 'country': 'UAE'}
        return {'city': 'Unknown', 'country': 'UAE'}
    
    # Check for other countries
    elif '-kuwait' in url:
        return {'city': 'Unknown', 'country': 'Kuwait'}
    elif '-qatar' in url:
        return {'city': 'Unknown', 'country': 'Qatar'}
    elif '-libya' in url:
        return {'city': 'Unknown', 'country': 'Libya'}
    elif '-germany' in url:
        return {'city': 'Unknown', 'country': 'Germany'}
    elif '-france' in url:
        return {'city': 'Unknown', 'country': 'France'}
    elif '-united-states' in url:
        return {'city': 'Unknown', 'country': 'United States'}
    
    return {'city': 'Unknown', 'country': 'Unknown'}


# Test the function
test_urls = [
    "https://wuzzuf.net/jobs/p/zav5d9coyr5v-procurement-analyst-guillam-alexandria-egypt",
    "https://wuzzuf.net/jobs/p/fsmj9wn13svm-backend-developer-chicago-united-states",
    "https://wuzzuf.net/jobs/p/commrc4uthhw-mid-senior-net-backend-developer-bedbank-sa-riyadh-saudi-arabia",
    "https://wuzzuf.net/jobs/p/xyz-cairo-egypt"
]

print("Testing URL location extraction:")
for url in test_urls:
    print(f"  {url.split('/')[-1][:50]}... → {extract_location_from_url(url)}")

Testing URL location extraction:
  zav5d9coyr5v-procurement-analyst-guillam-alexandri... → {'city': 'Alexandria', 'country': 'Egypt'}
  fsmj9wn13svm-backend-developer-chicago-united-stat... → {'city': 'Unknown', 'country': 'United States'}
  commrc4uthhw-mid-senior-net-backend-developer-bedb... → {'city': 'Riyadh', 'country': 'Saudi Arabia'}
  xyz-cairo-egypt... → {'city': 'Cairo', 'country': 'Egypt'}


In [38]:
# ============================================================
# STEP 19: Apply Location Extraction to Combined Dataset
# ============================================================

print("🧹 Extracting locations from URLs...")

# Apply location extraction to all records
location_parsed = combined_final_clean['job_url'].apply(extract_location_from_url)
combined_final_clean['city'] = location_parsed.apply(lambda x: x['city'])
combined_final_clean['country'] = location_parsed.apply(lambda x: x['country'])

# Update location_raw as well
combined_final_clean['location_raw'] = combined_final_clean.apply(
    lambda row: f"{row['city']}, {row['country']}" if row['city'] != 'Unknown' else 'Unknown',
    axis=1
)

print(f"\n✅ Location extraction complete!")

# Show results
print("\n📊 City Distribution (Top 15):")
print(combined_final_clean['city'].value_counts().head(15))

print("\n📊 Country Distribution:")
print(combined_final_clean['country'].value_counts())

🧹 Extracting locations from URLs...

✅ Location extraction complete!

📊 City Distribution (Top 15):
city
Cairo         435
Giza          104
Unknown        47
Alexandria     35
Riyadh          7
Port Said       4
Suez            4
Dubai           2
Aswan           2
Assiut          1
Beni Suef       1
Khobar          1
Name: count, dtype: int64

📊 Country Distribution:
country
Egypt            613
Saudi Arabia      13
Libya              5
Unknown            4
United States      3
UAE                3
Germany            1
France             1
Name: count, dtype: int64


In [39]:
# ============================================================
# STEP 20: Save Updated Combined Dataset
# ============================================================

# Save to CSV
output_path = "../data/cleaned/jobs_combined.csv"
combined_final_clean.to_csv(output_path, index=False, encoding='utf-8')

print(f"✅ Updated dataset saved to: {output_path}")
print(f"   Total records: {len(combined_final_clean)}")

# Also save JSON version
json_output_path = "../data/cleaned/jobs_combined.json"
json_df = combined_final_clean.copy()
json_df['skills_list'] = json_df['skills_list'].apply(lambda x: json.dumps(x) if isinstance(x, list) else x)
json_df.to_json(json_output_path, orient='records', indent=2, force_ascii=False)
print(f"✅ JSON version saved to: {json_output_path}")

# Final summary
print("\n" + "=" * 70)
print("📊 FINAL COMBINED DATASET - UPDATED WITH LOCATIONS")
print("=" * 70)

print(f"\n✅ Total Jobs: {len(combined_final_clean)}")

print(f"\n📊 Data Sources:")
print(combined_final_clean['data_source'].value_counts())

print(f"\n📊 Experience Levels:")
print(combined_final_clean['experience_level'].value_counts())

print(f"\n📊 Top 10 Job Categories:")
print(combined_final_clean['job_category'].value_counts().head(10))

print(f"\n📊 Top 10 Cities:")
print(combined_final_clean['city'].value_counts().head(10))

print(f"\n📊 Countries:")
print(combined_final_clean['country'].value_counts())

print(f"\n📁 Files saved:")
print("   • data/cleaned/jobs_combined.csv")
print("   • data/cleaned/jobs_combined.json")

print("\n✅ Ready for DBT & ETL pipelines!")

✅ Updated dataset saved to: ../data/cleaned/jobs_combined.csv
   Total records: 643
✅ JSON version saved to: ../data/cleaned/jobs_combined.json

📊 FINAL COMBINED DATASET - UPDATED WITH LOCATIONS

✅ Total Jobs: 643

📊 Data Sources:
data_source
csv_wuzzuf_jobs    578
json_wuzzuf         65
Name: count, dtype: int64

📊 Experience Levels:
experience_level
mid          289
senior       175
junior       114
executive     38
Name: count, dtype: int64

📊 Top 10 Job Categories:
job_category
data engineer             141
analytics engineer        103
data analyst               98
sql developer              83
ai engineer                42
power bi                   36
business analyst           25
backend developer          16
python developer data      16
data pipeline engineer     16
Name: count, dtype: int64

📊 Top 10 Cities:
city
Cairo         435
Giza          104
Unknown        47
Alexandria     35
Riyadh          7
Port Said       4
Suez            4
Dubai           2
Aswan           2
As

In [40]:
# ============================================================
# STEP 21: Extract Company Name from Job URLs
# ============================================================

# Let's examine some URLs to understand the pattern
sample_urls = combined_final_clean['job_url'].head(20).tolist()

print("Examining URL patterns to extract company names:")
for url in sample_urls[:10]:
    print(f"  {url}")

Examining URL patterns to extract company names:
  https://wuzzuf.net/jobs/p/f3fi6wjnx9o9-middle-data-engineer-vertex-technologies-cairo-egypt
  https://wuzzuf.net/jobs/p/dw9vbculsh2b-senior-data-engineer-vertex-technologies-cairo-egypt
  https://wuzzuf.net/jobs/p/beqgj9o9kpfk-data-engineer-hyundai-rotem-cairo-egypt
  https://wuzzuf.net/jobs/p/bitfhi15bemy-data-center-engineer-compute-storage-data-protection-cairo-egypt
  https://wuzzuf.net/jobs/p/mc8ehc6xddfq-infrastructure-network-operations-engineer-data-center-wan-cairo-egypt
  https://wuzzuf.net/jobs/p/x8tb3ijdep9x-senior-sales-engineer-data-centers-mission-critical-infrastructure-mobi-egypt-cairo-egypt
  https://wuzzuf.net/jobs/p/hif824t4axxr-data-scientist-and-machine-learning-engineer-applied-ai-knowledge-sharing-epsilonai-cairo-egypt
  https://wuzzuf.net/jobs/p/r1yxretv7hyo-ai-data-science-engineer-raya-distribution-cairo-egypt
  https://wuzzuf.net/jobs/p/laprqjalr4h6-senior-security-engineer-security-team-lead-anzemah-giza-eg

In [41]:
def extract_company_from_url(url: str, job_title: str = None) -> str:
    """
    Extract company name from job URL
    Example: https://wuzzuf.net/jobs/p/f3fi6wjnx9o9-middle-data-engineer-vertex-technologies-cairo-egypt
             → company='Vertex Technologies'
    """
    if pd.isna(url) or not url:
        return 'Unknown'
    
    url = str(url)
    
    # Get the last part of URL (before the location)
    # Pattern: ...-company-city-country
    parts = url.split('/')[-1]  # Get the job ID part
    
    # Split by '-' and get parts
    url_parts = parts.split('-')
    
    # The company name is typically between job title and location
    # We need to identify where the job title ends and company begins
    
    # Known locations to identify where company name ends
    locations = [
        'cairo', 'egypt', 'giza', 'alexandria', 'mansoura', 'tanta', 'zagazig',
        'suez', 'ismailia', 'faiyum', 'beni-suef', 'minya', 'assiut', 'sohag',
        'qena', 'luxor', 'aswan', 'damietta', 'port-said', 'new-cairo', 
        'new capital', 'maadi', 'nasr city', 'heliopolis', 'mohandessin', 
        'dokki', '6th of october', 'obour city', 'shorouk city', 'badr city',
        'sheraton', 'abbassia', 'riyadh', 'saudi arabia', 'sa', 'jeddah',
        'mecca', 'dammam', 'khobar', 'dubai', 'uae', 'united arab emirates',
        'kuwait', 'qatar', 'libya', 'germany', 'france', 'united states'
    ]
    
    # Find where location starts
    company_parts = []
    for i, part in enumerate(url_parts):
        # Check if this part is a location
        is_location = False
        for loc in locations:
            if part.lower() == loc or part.lower() == loc.replace(' ', '-'):
                is_location = True
                break
        
        if not is_location and i > 0:  # Skip the first part (job_id)
            # Check if it's not a common job title word
            job_words = ['junior', 'senior', 'mid', 'level', 'lead', 'principal',
                        'engineer', 'developer', 'analyst', 'manager', 'specialist',
                        'associate', 'intern', 'data', 'business', 'software', 
                        'backend', 'frontend', 'full', 'stack', 'machine', 'learning',
                        'artificial', 'intelligence', 'ai', 'procurement', 'sales',
                        'marketing', 'financial', 'accounting', 'investment', 'process']
            if part.lower() not in job_words:
                company_parts.append(part)
    
    if company_parts:
        company = ' '.join(company_parts)
        # Clean up
        company = company.title()
        # Remove trailing numbers if any
        import re
        company = re.sub(r'\d+$', '', company).strip()
        if company:
            return company
    
    return 'Unknown'


# Test the function
test_urls = [
    "https://wuzzuf.net/jobs/p/f3fi6wjnx9o9-middle-data-engineer-vertex-technologies-cairo-egypt",
    "https://wuzzuf.net/jobs/p/beqgj9o9kpfk-data-engineer-hyundai-rotem-cairo-egypt",
    "https://wuzzuf.net/jobs/p/x8tb3ijdep9x-senior-sales-engineer-data-centers-mission-critical-infrastructure-mobi-egypt-cairo-egypt",
    "https://wuzzuf.net/jobs/p/hif824t4axxr-data-scientist-and-machine-learning-engineer-applied-ai-knowledge-sharing-epsilonai-cairo-egypt"
]

print("Testing company extraction from URLs:")
for url in test_urls:
    company = extract_company_from_url(url)
    print(f"  {url.split('/')[-1][:60]}...")
    print(f"    → Company: {company}")

Testing company extraction from URLs:
  f3fi6wjnx9o9-middle-data-engineer-vertex-technologies-cairo-...
    → Company: Middle Vertex Technologies
  beqgj9o9kpfk-data-engineer-hyundai-rotem-cairo-egypt...
    → Company: Hyundai Rotem
  x8tb3ijdep9x-senior-sales-engineer-data-centers-mission-crit...
    → Company: Centers Mission Critical Infrastructure Mobi
  hif824t4axxr-data-scientist-and-machine-learning-engineer-ap...
    → Company: Scientist And Applied Knowledge Sharing Epsilonai


In [42]:
# Better approach: Use job_title to identify where it ends in URL
def extract_company_from_url_v2(url: str, job_title: str = None) -> str:
    """
    Extract company name from job URL by identifying job title position
    """
    if pd.isna(url) or not url:
        return 'Unknown'
    
    url = str(url)
    url_path = url.split('/')[-1]  # Get the job ID part
    url_parts = url_path.split('-')
    
    # Known locations to identify where company name ends
    locations = [
        'cairo', 'egypt', 'giza', 'alexandria', 'mansoura', 'tanta', 'zagazig',
        'suez', 'ismailia', 'faiyum', 'beni-suef', 'minya', 'assiut', 'sohag',
        'qena', 'luxor', 'aswan', 'damietta', 'port-said', 'new-cairo', 
        'new capital', 'maadi', 'nasr city', 'heliopolis', 'mohandessin', 
        'dokki', '6th of october', 'obour city', 'shorouk city', 'badr city',
        'sheraton', 'abbassia', 'riyadh', 'saudi arabia', 'sa', 'jeddah',
        'mecca', 'dammam', 'khobar', 'dubai', 'uae', 'united arab emirates',
        'kuwait', 'qatar', 'libya', 'germany', 'france', 'united states'
    ]
    
    # Find where location starts (from the end)
    location_idx = -1
    for i in range(len(url_parts) - 1, -1, -1):
        if url_parts[i].lower() in locations:
            location_idx = i
            break
    
    if location_idx == -1:
        return 'Unknown'
    
    # Company is between job_title (parts 1 to location_idx-1)
    # But we need to skip the job_id (part 0)
    # Job title typically has 1-4 parts like: "data-engineer", "senior-data-engineer"
    
    # Find job title end by looking for common job words
    job_words = {'engineer', 'developer', 'analyst', 'manager', 'specialist',
                'associate', 'intern', 'scientist', 'architect', 'lead',
                'director', 'head', 'chief', 'consultant', 'expert', 'officer'}
    
    # Start after job_id (index 0), look for job title end
    job_title_end = 1
    for i in range(1, location_idx):
        if url_parts[i].lower() in job_words:
            job_title_end = i + 1
            break
    
    # Company is between job_title_end and location_idx
    company_parts = url_parts[job_title_end:location_idx]
    
    if company_parts:
        company = ' '.join(company_parts)
        company = company.title()
        if company:
            return company
    
    return 'Unknown'


# Test the improved function
print("Testing improved company extraction:")
for url in test_urls:
    company = extract_company_from_url_v2(url)
    print(f"  → {company}")

Testing improved company extraction:
  → Vertex Technologies Cairo
  → Hyundai Rotem Cairo
  → Data Centers Mission Critical Infrastructure Mobi Egypt Cairo
  → And Machine Learning Engineer Applied Ai Knowledge Sharing Epsilonai Cairo


In [43]:
# Smart approach: Use job_title to find position in URL
def extract_company_from_url_v3(url: str, job_title: str = None) -> str:
    """
    Extract company name from job URL by matching job title position
    """
    if pd.isna(url) or not url:
        return 'Unknown'
    
    url = str(url)
    url_path = url.split('/')[-1]  # Get the job ID part
    url_parts = url_path.split('-')
    
    # Known locations
    locations = [
        'cairo', 'egypt', 'giza', 'alexandria', 'mansoura', 'tanta', 'zagazig',
        'suez', 'ismailia', 'faiyum', 'beni-suef', 'minya', 'assiut', 'sohag',
        'qena', 'luxor', 'aswan', 'damietta', 'port-said', 'new-cairo', 
        'new capital', 'maadi', 'nasr city', 'heliopolis', 'mohandessin', 
        'dokki', '6th of october', 'obour city', 'shorouk city', 'badr city',
        'sheraton', 'abbassia', 'riyadh', 'saudi arabia', 'sa', 'jeddah',
        'mecca', 'dammam', 'khobar', 'dubai', 'uae', 'united arab emirates',
        'kuwait', 'qatar', 'libya', 'germany', 'france', 'united states'
    ]
    
    # Find location index from end
    location_idx = -1
    for i in range(len(url_parts) - 1, -1, -1):
        if url_parts[i].lower() in locations:
            location_idx = i
            break
    
    if location_idx == -1:
        return 'Unknown'
    
    # If we have job_title, use it to find where job title ends
    if job_title and pd.notna(job_title):
        # Normalize job title for matching
        title_words = str(job_title).lower().split()
        
        # Find where job title appears in URL parts
        title_end_idx = -1
        for i in range(1, min(location_idx, 6)):  # Job title typically 1-5 parts
            # Check if this part matches any title word
            for tw in title_words:
                if tw in url_parts[i].lower():
                    title_end_idx = i
                    break
            if title_end_idx > 0:
                break
        
        if title_end_idx > 0:
            # Company is between title_end_idx+1 and location_idx
            company_parts = url_parts[title_end_idx+1:location_idx]
            if company_parts:
                company = ' '.join(company_parts)
                company = company.title()
                if company:
                    return company
    
    # Fallback: assume company is the 2-3 parts after job_id
    # Typical pattern: job_id-job_title-company-city-country
    company_parts = url_parts[1:min(4, location_idx)]
    if company_parts:
        company = ' '.join(company_parts)
        company = company.title()
        if company:
            return company
    
    return 'Unknown'


# Test with sample data
sample_df = combined_final_clean[['job_title', 'job_url']].head(10)
print("Testing with actual job titles:")
for _, row in sample_df.iterrows():
    company = extract_company_from_url_v3(row['job_url'], row['job_title'])
    print(f"  {row['job_title'][:40]:<40} → {company}")

Testing with actual job titles:
  Middle Data Engineer                     → Data Engineer Vertex Technologies Cairo
  Senior Data Engineer                     → Data Engineer Vertex Technologies Cairo
  Data Engineer                            → Engineer Hyundai Rotem Cairo
  Data Center Engineer (Compute, Storage & → Center Engineer Compute Storage Data Protection Cairo
  Infrastructure Network Operations Engine → Network Operations Engineer Data Center Wan Cairo
  Senior Sales Engineer – Data Centers & M → Sales Engineer Data Centers Mission Critical Infrastructure Mobi Egypt Cairo
  Data Scientist And Machine learning Engi → Scientist And Machine Learning Engineer Applied Ai Knowledge Sharing Epsilonai Cairo
  AI & Data Science Engineer               → Data Science Engineer Raya Distribution Cairo
  Senior Security Engineer / Security Team → Security Engineer Security Team Lead Anzemah Giza
  Cost Estimation Engineer (procurement)   → Estimation Engineer Procurement Aswaq Cairo


In [44]:
# More robust approach: Find job title words in URL
def extract_company_final(url: str, job_title: str = None) -> str:
    """
    Extract company name from job URL - robust version
    """
    if pd.isna(url) or not url:
        return 'Unknown'
    
    url = str(url)
    url_path = url.split('/')[-1]
    url_parts = url_path.split('-')
    
    # Known locations
    locations = [
        'cairo', 'egypt', 'giza', 'alexandria', 'mansoura', 'tanta', 'zagazig',
        'suez', 'ismailia', 'faiyum', 'beni-suef', 'minya', 'assiut', 'sohag',
        'qena', 'luxor', 'aswan', 'damietta', 'port-said', 'new-cairo', 
        'new capital', 'maadi', 'nasr city', 'heliopolis', 'mohandessin', 
        'dokki', '6th of october', 'obour city', 'shorouk city', 'badr city',
        'sheraton', 'abbassia', 'riyadh', 'saudi arabia', 'sa', 'jeddah',
        'mecca', 'dammam', 'khobar', 'dubai', 'uae', 'united arab emirates',
        'kuwait', 'qatar', 'libya', 'germany', 'france', 'united states'
    ]
    
    # Find location index from end
    location_idx = -1
    for i in range(len(url_parts) - 1, -1, -1):
        if url_parts[i].lower() in locations:
            location_idx = i
            break
    
    if location_idx == -1:
        return 'Unknown'
    
    # Common job title words to identify where title ends
    job_words = {'junior', 'senior', 'mid', 'level', 'lead', 'principal',
                'engineer', 'developer', 'analyst', 'manager', 'specialist',
                'associate', 'intern', 'data', 'business', 'software', 
                'backend', 'frontend', 'full', 'stack', 'machine', 'learning',
                'artificial', 'intelligence', 'ai', 'scientist', 'architect',
                'procurement', 'sales', 'marketing', 'financial', 'accounting',
                'investment', 'process', 'security', 'network', 'operations',
                'estimation', 'cost', 'center', 'infrastructure', 'storage',
                'protection', 'wan', 'team', 'head', 'chief', 'consultant',
                'expert', 'officer', 'director', 'vp', 'vice', 'president'}
    
    # Find where job title ends (last job word before location)
    title_end = 1
    for i in range(1, location_idx):
        if url_parts[i].lower() in job_words:
            title_end = i + 1
    
    # Company is between title_end and location_idx
    company_parts = url_parts[title_end:location_idx]
    
    if company_parts:
        company = ' '.join(company_parts)
        company = company.title()
        # Clean up common issues
        company = company.replace('And ', '').replace(' Of ', ' ')
        if company:
            return company
    
    return 'Unknown'


# Test with more samples
sample_df = combined_final_clean[['job_title', 'job_url']].head(15)
print("Testing final company extraction:")
for _, row in sample_df.iterrows():
    company = extract_company_final(row['job_url'], row['job_title'])
    print(f"  {row['job_title'][:35]:<35} → {company}")

Testing final company extraction:
  Middle Data Engineer                → Vertex Technologies Cairo
  Senior Data Engineer                → Vertex Technologies Cairo
  Data Engineer                       → Hyundai Rotem Cairo
  Data Center Engineer (Compute, Stor → Cairo
  Infrastructure Network Operations E → Cairo
  Senior Sales Engineer – Data Center → Mobi Egypt Cairo
  Data Scientist And Machine learning → Knowledge Sharing Epsilonai Cairo
  AI & Data Science Engineer          → Raya Distribution Cairo
  Senior Security Engineer / Security → Anzemah Giza
  Cost Estimation Engineer (procureme → Aswaq Cairo
  Traffic Engineer                    → Seero Cairo
  Senior Sales Engineer               → Mobi Egypt Cairo
  AI Engineer                         → Cairo
  Mechanical Production Engineer      → Ilock Dakahlia
  Production Engineer                 → El Obour For Paper Production Cairo


In [45]:
# Fixed: Find the LAST job word before location
def extract_company_v4(url: str, job_title: str = None) -> str:
    """
    Extract company name from job URL - fixed version
    """
    if pd.isna(url) or not url:
        return 'Unknown'
    
    url = str(url)
    url_path = url.split('/')[-1]
    url_parts = url_path.split('-')
    
    # Known locations
    locations = [
        'cairo', 'egypt', 'giza', 'alexandria', 'mansoura', 'tanta', 'zagazig',
        'suez', 'ismailia', 'faiyum', 'beni-suef', 'minya', 'assiut', 'sohag',
        'qena', 'luxor', 'aswan', 'damietta', 'port-said', 'new-cairo', 
        'new capital', 'maadi', 'nasr city', 'heliopolis', 'mohandessin', 
        'dokki', '6th of october', 'obour city', 'shorouk city', 'badr city',
        'sheraton', 'abbassia', 'riyadh', 'saudi arabia', 'sa', 'jeddah',
        'mecca', 'dammam', 'khobar', 'dubai', 'uae', 'united arab emirates',
        'kuwait', 'qatar', 'libya', 'germany', 'france', 'united states'
    ]
    
    # Find location index from end
    location_idx = -1
    for i in range(len(url_parts) - 1, -1, -1):
        if url_parts[i].lower() in locations:
            location_idx = i
            break
    
    if location_idx == -1:
        return 'Unknown'
    
    # Core job title words (most common)
    core_job_words = {'engineer', 'developer', 'analyst', 'manager', 'specialist',
                     'scientist', 'architect', 'consultant', 'director', 'lead',
                     'head', 'chief', 'officer', 'intern', 'associate'}
    
    # Find the LAST core job word before location (this is where title ends)
    title_end = 1
    for i in range(location_idx - 1, 0, -1):
        if url_parts[i].lower() in core_job_words:
            title_end = i + 1
            break
    
    # Company is between title_end and location_idx
    company_parts = url_parts[title_end:location_idx]
    
    if company_parts:
        company = ' '.join(company_parts)
        company = company.title()
        # Clean up
        company = company.replace('And ', '').replace(' Of ', ' ')
        if company and len(company) > 1:
            return company
    
    return 'Unknown'


# Test again
sample_df = combined_final_clean[['job_title', 'job_url']].head(15)
print("Testing fixed company extraction:")
for _, row in sample_df.iterrows():
    company = extract_company_v4(row['job_url'], row['job_title'])
    print(f"  {row['job_title'][:35]:<35} → {company}")

Testing fixed company extraction:
  Middle Data Engineer                → Vertex Technologies Cairo
  Senior Data Engineer                → Vertex Technologies Cairo
  Data Engineer                       → Hyundai Rotem Cairo
  Data Center Engineer (Compute, Stor → Compute Storage Data Protection Cairo
  Infrastructure Network Operations E → Data Center Wan Cairo
  Senior Sales Engineer – Data Center → Data Centers Mission Critical Infrastructure Mobi Egypt Cairo
  Data Scientist And Machine learning → Applied Ai Knowledge Sharing Epsilonai Cairo
  AI & Data Science Engineer          → Raya Distribution Cairo
  Senior Security Engineer / Security → Anzemah Giza
  Cost Estimation Engineer (procureme → Procurement Aswaq Cairo
  Traffic Engineer                    → Seero Cairo
  Senior Sales Engineer               → Mobi Egypt Cairo
  AI Engineer                         → Cairo
  Mechanical Production Engineer      → Ilock Dakahlia
  Production Engineer                 → El Obour For Pap

In [51]:
import pandas as pd
import re

def extract_company_robust(url: str, job_title: str) -> str:
    """
    Extracts the company name located between the job title and the location in Wuzzuf URLs.
    """
    if pd.isna(url) or not url or 'wuzzuf.net' not in str(url):
        return 'Unknown'
    
    # 1. Get the slug (the text after the last /)
    slug = str(url).split('/')[-1].lower()
    url_parts = slug.split('-')
    
    # 2. Convert job title to words for skipping
    title_slug = re.sub(r'[^a-z0-9]+', '-', str(job_title).lower()).strip('-')
    title_parts = title_slug.split('-')
    
    # 3. Skip the Wuzzuf ID and all words that are part of the Job Title
    start_idx = 1
    for i in range(1, len(url_parts)):
        if url_parts[i] in title_parts:
            start_idx = i + 1
        else:
            if start_idx > 1: break 

    # 4. List of locations to stop the extraction
    locations = {
        'egypt', 'cairo', 'giza', 'alexandria', 'maadi', 'nasr', 'city', '6th', 
        'october', 'obour', 'helwan', 'sharqia', 'mansoura', 'tanta', 'assiut', 
        'sohag', 'suez', 'ismailia', 'qena', 'luxor', 'aswan', 'hurghada', 
        'sharm', 'sheikh', 'zayed', 'rehab', 'badr', 'shorouk', 'dokki', 'zamalek',
        'faisal', 'haram', 'shubra', 'tagamoa', 'new'
    }
    
    # 5. Extract words until a location keyword is hit
    remaining_parts = url_parts[start_idx:]
    company_parts = []
    for part in remaining_parts:
        if part in locations:
            break
        company_parts.append(part)
        
    # 6. Format and Return
    if not company_parts:
        return "Confidential"
        
    company = " ".join(company_parts).title()
    # Clean up common business noise
    return company.replace('And ', '& ').replace(' Of ', ' of ').replace('The ', '')

# --- UPDATE YOUR DATASET ---

# Load the combined file
df_combined = pd.read_csv('../data/cleaned/jobs_combined.csv')

# Target only the 'Unknown' entries
mask = (df_combined['company_name'] == 'Unknown')
print(f"Cleaning {mask.sum()} company names...")

df_combined.loc[mask, 'company_name'] = df_combined.loc[mask].apply(
    lambda x: extract_company_robust(x['job_url'], x['job_title']), 
    axis=1
)

# Save back to CSV
df_combined.to_csv('../data/cleaned/jobs_combined.csv', index=False)
print("✅ Done! File 'jobs_combined.csv' is updated.")

Cleaning 587 company names...
✅ Done! File 'jobs_combined.csv' is updated.


In [ ]:
import pandas as pd
import os

# 1. Define the correct path to your cleaned file
path = "../data/cleaned/" 
file_name = "jobs_combined.csv"
full_path = os.path.join(path, file_name)

# 2. Check if the file exists and load it
if os.path.exists(full_path):
    df_combined = pd.read_csv(full_path)
    
    # 3. Remove the column
    if 'posted_date' in df_combined.columns:
        df_combined.drop(columns=['posted_date'], inplace=True)
        print("🗑️ Column 'posted_date' has been removed from the DataFrame.")
        
        # 4. Save the updated file
        try:
            df_combined.to_csv(full_path, index=False)
            print(f"✅ Success! Updated file saved at: {full_path}")
        except PermissionError:
            print("❌ Permission Denied: Please close the CSV file in Excel and run this cell again.")
    else:
        print("⚠️ Column 'posted_date' not found in the dataset. It might already be removed.")
else:
    print(f"❌ Error: Could not find the file at {full_path}")

In [2]:
import pandas as pd
import os

# 1. Path setup (same as your previous successful one)
path = "../data/cleaned/" 
file_name = "jobs_combined.csv"
full_path = os.path.join(path, file_name)

# 2. Load the data
if os.path.exists(full_path):
    df = pd.read_csv(full_path)
    
    def extract_clean_date(text):
        if pd.isna(text):
            return text
        
        # Convert to string and split by new lines
        text_str = str(text)
        if '\n' in text_str:
            # The date is usually the last line (e.g., "2 days ago")
            lines = [line.strip() for line in text_str.split('\n') if line.strip()]
            final_line = lines[-1]
            
            # Standardize: make sure it says "posted ..." to match the rest of the CSV
            if "posted" not in final_line.lower():
                return f"posted {final_line}"
            return final_line
        
        return text

    # 3. Apply ONLY to the first 66 rows (index 0 to 65)
    print("Extracting dates from text blocks in the first 66 rows...")
    df.loc[0:65, 'posted_date'] = df.loc[0:65, 'posted_date'].apply(extract_clean_date)

    # 4. Save the file (MAKE SURE EXCEL IS CLOSED!)
    try:
        df.to_csv(full_path, index=False)
        print("✅ Success! The posted_date column is now perfectly cleaned.")
        
        # Verify the top rows
        print("\nVerification (First 5 rows):")
        display(df[['job_title', 'posted_date']].head())
    except PermissionError:
        print("❌ Permission Denied: Please close 'jobs_combined.csv' in Excel and try again!")

else:
    print(f"❌ Error: Could not find the file at {full_path}")

Extracting dates from text blocks in the first 66 rows...
✅ Success! The posted_date column is now perfectly cleaned.

Verification (First 5 rows):


,job_title,posted_date
0,Middle Data Engineer,posted 2 days ago
1,Senior Data Engineer,posted 2 days ago
2,Data Engineer,posted 19 days ago
3,"Data Center Engineer (Compute, Storage & Data ...",posted 20 days ago
4,Infrastructure Network Operations Engineer (Da...,posted 20 days ago


In [3]:
import pandas as pd
import os

# 1. Path setup
path = "../data/cleaned/" 
file_name = "jobs_combined.csv"
full_path = os.path.join(path, file_name)

df = pd.read_csv(full_path)

# 2. Define the Mapping
# This maps districts back to their parent Governorate
district_to_city = {
    'Maadi': 'Cairo',
    'Nasr City': 'Cairo',
    'New Cairo': 'Cairo',
    'Heliopolis': 'Cairo',
    'Sheraton': 'Cairo',
    'Zamalek': 'Cairo',
    'Downtown': 'Cairo',
    'Shubra': 'Cairo',
    'Rehab': 'Cairo',
    'Madinaty': 'Cairo',
    'Dokki': 'Giza',
    'Mohandessin': 'Giza',
    '6th Of October': 'Giza',
    'Sheikh Zayed': 'Giza',
    'Faisal': 'Giza',
    'Haram': 'Giza',
    'Smart Village': 'Giza'
}

def clean_location_hierarchy(row):
    current_loc = str(row['city']).title().strip()
    
    # Check if the value is actually a district we know
    if current_loc in district_to_city:
        return district_to_city[current_loc], current_loc
    
    # If it's already a main city, return it as city and leave district empty
    return current_loc, "Other/Main"

# 3. Apply the logic to create TWO columns
print("🗺️ Creating Location Hierarchy...")
df[['governorate', 'district']] = df.apply(
    lambda x: pd.Series(clean_location_hierarchy(x)), axis=1
)

# 4. Final Cleanup: Drop the old 'city' and 'country' columns
cols_to_drop = [c for c in ['city', 'country'] if c in df.columns]
df.drop(columns=cols_to_drop, inplace=True)

# 5. Save
try:
    df.to_csv(full_path, index=False)
    print("✅ Success! Country dropped. Location split into 'governorate' and 'district'.")
    display(df[['governorate', 'district']].head(10))
except PermissionError:
    print("❌ Close Excel!")

🗺️ Creating Location Hierarchy...
✅ Success! Country dropped. Location split into 'governorate' and 'district'.


,governorate,district
0,Cairo,Other/Main
1,Cairo,Other/Main
2,Cairo,Other/Main
3,Cairo,Other/Main
4,Cairo,Other/Main
5,Cairo,Other/Main
6,Cairo,Other/Main
7,Cairo,Other/Main
8,Giza,Other/Main
9,Cairo,Other/Main


In [5]:
import pandas as pd
import os
import re

# 1. Path setup
path = "../data/cleaned/" 
file_name = "jobs_combined.csv"
full_path = os.path.join(path, file_name)

df = pd.read_csv(full_path)

def extract_location_from_url(row):
    # If we already have a valid governorate that isn't 'Unknown', keep it
    current_gov = str(row.get('governorate', 'Unknown'))
    if current_gov != 'Unknown' and pd.notna(current_gov):
        return current_gov
    
    url = str(row['job_url']).lower()
    
    # regex to find the word before -egypt at the end of the URL
    # Example: ...dakahlia-egypt -> pulls 'dakahlia'
    match = re.search(r'-([a-z0-9-]+)-egypt', url)
    
    if match:
        extracted = match.group(1).split('-')[-1] # Get the last part if it's like 'new-cairo'
        return extracted.title()
    
    return current_gov

# 2. Apply the extraction
print("🔎 Extracting missing locations from URL patterns...")
df['governorate'] = df.apply(extract_location_from_url, axis=1)

# 3. Quick cleanup for specific cases
# Sometimes 'new-cairo' or '6th-of-october' gets cut, let's fix the common ones
mapping_fixes = {
    'Cairo': 'Cairo',
    'Giza': 'Giza',
    'Dakahlia': 'Dakahlia',
    'Alexandria': 'Alexandria',
    'October': 'Giza', # Mapping 6th of October to Giza
    'Zayed': 'Giza'     # Mapping Sheikh Zayed to Giza
}

df['governorate'] = df['governorate'].replace(mapping_fixes)

# 4. Save (Close Excel!)
try:
    df.to_csv(full_path, index=False)
    print("✅ Success! Missing locations backfilled from URLs.")
    # Show the results where it was likely unknown before
    display(df[['job_title', 'governorate', 'job_url']].iloc[70:80])
except PermissionError:
    print("❌ Close Excel!")

🔎 Extracting missing locations from URL patterns...
✅ Success! Missing locations backfilled from URLs.


,job_title,governorate,job_url
70,Business Analyst / (Assiut),Assiut,https://wuzzuf.net/jobs/p/87gum36mxt9q-busines...
71,Accounting Analyst,Alexandria,https://wuzzuf.net/jobs/p/uwcau3s4x2dn-account...
72,Senior Treasury Analyst,Cairo,https://wuzzuf.net/jobs/p/kc0pb9zirkdm-senior-...
73,Sales Analyst,Cairo,https://wuzzuf.net/jobs/p/rv9cja0njfqe-sales-a...
74,Financial Planning Analyst,Cairo,https://wuzzuf.net/jobs/p/613uaj8nbpfw-financi...
75,Development Analyst / Business Analyst,Giza,https://wuzzuf.net/jobs/p/1ydyska6vimm-develop...
76,Sales Operations Analyst,Cairo,https://wuzzuf.net/jobs/p/sryjgwe6rblp-sales-o...
77,Procurement Analyst,Alexandria,https://wuzzuf.net/jobs/p/zav5d9coyr5v-procure...
78,Senior Investment Analyst,Giza,https://wuzzuf.net/jobs/p/yg1lvbhbvs2r-senior-...
79,Data Analyst,Cairo,https://wuzzuf.net/jobs/p/aoeklax6waer-data-an...


In [6]:
import pandas as pd
import os

# 1. Path setup
path = "../data/cleaned/" 
file_name = "jobs_combined.csv"
full_path = os.path.join(path, file_name)

df = pd.read_csv(full_path)

# 2. List of valid Egyptian Governorates to keep
# We'll keep rows that match these or were recovered earlier
egypt_govs = [
    'Cairo', 'Giza', 'Alexandria', 'Dakahlia', 'Red Sea', 'Beheira', 'Fayoum',
    'Gharbia', 'Ismailia', 'Monufia', 'Minya', 'Qalyubia', 'New Valley', 'Suez',
    'Aswan', 'Assiut', 'Beni Suef', 'Port Said', 'Damietta', 'Sharkia', 'South Sinai',
    'Kafr El Sheikh', 'Matrouh', 'Luxor', 'Qena', 'Sohag', 'North Sinai'
]

# 3. Perform the Filter
initial_count = len(df)

# Filter logic: 
# Keep only if it's in our Egypt list AND is not 'Unknown'
df_cleaned = df[df['governorate'].isin(egypt_govs)].copy()

# 4. Explicitly drop mentioned international cities (just to be safe)
international_cities = ['Dubai', 'Riyadh', 'Khobar', 'Jeddah', 'Abu Dhabi', 'Doha', 'Kuwait City']
df_cleaned = df_cleaned[~df_cleaned['governorate'].isin(international_cities)]

final_count = len(df_cleaned)

# 5. Save the results
try:
    df_cleaned.to_csv(full_path, index=False)
    print(f"🧹 Data Purge Complete!")
    print(f"📉 Initial rows: {initial_count}")
    print(f"📈 Final Egypt-only rows: {final_count}")
    print(f"🚫 Removed {initial_count - final_count} international/unknown rows.")
    
    # Check the unique values left to ensure it's pure
    print("\n🌍 Remaining Governorates in your dataset:")
    print(df_cleaned['governorate'].unique())
    
except PermissionError:
    print("❌ Close Excel!")

🧹 Data Purge Complete!
📉 Initial rows: 643
📈 Final Egypt-only rows: 590
🚫 Removed 53 international/unknown rows.

🌍 Remaining Governorates in your dataset:
<StringArray>
[     'Cairo',       'Giza',   'Dakahlia', 'Alexandria',     'Assiut',
  'Port Said',    'Beheira',      'Aswan',       'Suez',    'Red Sea',
  'Beni Suef']
Length: 11, dtype: str


In [7]:
import pandas as pd
import os

# 1. Path setup
path = "../data/cleaned/" 
file_name = "jobs_combined.csv"
full_path = os.path.join(path, file_name)

df = pd.read_csv(full_path)

# 2. Expanded Mapping
district_to_gov = {
    # Cairo Districts
    'Maadi': 'Cairo', 'Nasr City': 'Cairo', 'New Cairo': 'Cairo', 'Heliopolis': 'Cairo',
    'Sheraton': 'Cairo', 'Zamalek': 'Cairo', 'Downtown': 'Cairo', 'Shubra': 'Cairo',
    'Rehab': 'Cairo', 'Madinaty': 'Cairo', 'Shorouk': 'Cairo', 'Obour': 'Cairo',
    # Giza Districts
    'Dokki': 'Giza', 'Mohandessin': 'Giza', '6th Of October': 'Giza', 'Sheikh Zayed': 'Giza',
    'Faisal': 'Giza', 'Haram': 'Giza', 'Smart Village': 'Giza', 'Agouza': 'Giza'
}

# List of main governorates to check against
main_govs = ['Cairo', 'Giza', 'Alexandria', 'Dakahlia', 'Sharkia', 'Gharbia', 'Monufia', 'Suez']

def refine_location(row):
    # Use the 'governorate' we extracted earlier as a starting point
    gov = str(row['governorate']).title().strip()
    
    # If the governorate name itself is actually a district (like 'Maadi')
    if gov in district_to_gov:
        actual_gov = district_to_gov[gov]
        district = gov
        return actual_gov, district
    
    # If it's a main governorate, we look for a district in the 'job_url' or other columns
    # But for now, let's just make sure we aren't overwriting real data with "Other/Main"
    if gov in main_govs:
        return gov, "Main"

    return gov, "Other"

# 3. Re-apply the logic
print("🔄 Refining District data...")
df[['governorate', 'district']] = df.apply(
    lambda x: pd.Series(refine_location(x)), axis=1
)

# 4. Save
try:
    df.to_csv(full_path, index=False)
    print("✅ Success! Districts are now more specific.")
    print("\nSample of your new location hierarchy:")
    display(df[['governorate', 'district']].value_counts().head(10))
except PermissionError:
    print("❌ Close Excel!")

🔄 Refining District data...
✅ Success! Districts are now more specific.

Sample of your new location hierarchy:


governorate  district
Cairo        Main        435
Giza         Main        104
Alexandria   Main         35
Port Said    Other         4
Suez         Main          4
Dakahlia     Main          2
Aswan        Other         2
Assiut       Other         1
Beheira      Other         1
Red Sea      Other         1
Name: count, dtype: int64

In [8]:
import pandas as pd
import os
import re

# 1. Path setup
path = "../data/cleaned/" 
file_name = "jobs_combined.csv"
full_path = os.path.join(path, file_name)

df = pd.read_csv(full_path)

# 2. Rename the column as requested
if 'governorate' in df.columns:
    df.rename(columns={'governorate': 'city'}, inplace=True)

# 3. Improved Extraction Logic
def extract_district_properly(row):
    url = str(row['job_url']).lower()
    
    # List of common districts to look for in the URL
    districts = [
        'maadi', 'nasr-city', 'new-cairo', 'heliopolis', 'sheraton', 'zamalek', 
        'rehab', 'madinaty', 'shorouk', 'obour', 'dokki', 'mohandessin', 
        '6th-of-october', 'sheikh-zayed', 'faisal', 'haram', 'smart-village', 
        'agouza', 'tagamoa', 'fifth-settlement'
    ]
    
    for d in districts:
        if d in url:
            return d.replace('-', ' ').title()
            
    # If no specific district is found in the URL, return what was already there
    return row.get('district', 'Main')

# 4. Apply the extraction
print("🧹 Renaming columns and digging for Districts...")
df['district'] = df.apply(extract_district_properly, axis=1)

# 5. Clean up the 'City' column to ensure it's not 'Other'
# If district is 'Maadi', city should definitely be 'Cairo'
mapping = {
    'Maadi': 'Cairo', 'Nasr City': 'Cairo', 'New Cairo': 'Cairo', 
    'Tagamoa': 'Cairo', 'Fifth Settlement': 'Cairo', 'Heliopolis': 'Cairo',
    'Dokki': 'Giza', 'Mohandessin': 'Giza', '6th Of October': 'Giza', 
    'Sheikh Zayed': 'Giza', 'Smart Village': 'Giza'
}

for district, city_name in mapping.items():
    df.loc[df['district'] == district, 'city'] = city_name

# 6. Save (Close Excel!)
try:
    df.to_csv(full_path, index=False)
    print("✅ Success! 'governorate' is now 'city'.")
    print("✅ Districts have been extracted from URLs where possible.")
    
    # Show the breakdown to verify
    print("\nValue counts for Districts:")
    print(df['district'].value_counts().head(10))
except PermissionError:
    print("❌ Close Excel!")

🧹 Renaming columns and digging for Districts...
✅ Success! 'governorate' is now 'city'.
✅ Districts have been extracted from URLs where possible.

Value counts for Districts:
district
Main              576
Other              10
Obour               3
6Th Of October      1
Name: count, dtype: int64


In [9]:
import pandas as pd
import os

# 1. Path setup
path = "../data/cleaned/" 
file_name = "jobs_combined.csv"
full_path = os.path.join(path, file_name)

df = pd.read_csv(full_path)

# 2. Define the list of valid Egyptian cities/governorates
egypt_cities = [
    'Cairo', 'Giza', 'Alexandria', 'Dakahlia', 'Red Sea', 'Beheira', 'Fayoum',
    'Gharbia', 'Ismailia', 'Monufia', 'Minya', 'Qalyubia', 'New Valley', 'Suez',
    'Aswan', 'Assiut', 'Beni Suef', 'Port Said', 'Damietta', 'Sharkia', 'South Sinai',
    'Kafr El Sheikh', 'Matrouh', 'Luxor', 'Qena', 'Sohag', 'North Sinai'
]

# 3. Last-chance recovery from URL before dropping
def final_city_check(row):
    city = str(row.get('city', 'Unknown')).strip()
    url = str(row['job_url']).lower()
    
    # If it's already a valid city, keep it
    if city in egypt_cities:
        return city
    
    # Check the URL for any Egyptian city one last time
    for c in egypt_cities:
        if c.lower() in url:
            return c
            
    return "To Drop"

# 4. Apply recovery and Filter
print("🧹 Finalizing City column and purging non-Egypt rows...")
df['city'] = df.apply(final_city_check, axis=1)

# Keep only the rows that aren't "To Drop"
initial_rows = len(df)
df = df[df['city'] != "To Drop"].copy()

# 5. Remove the extra columns
cols_to_remove = ['district', 'location_raw', 'location', 'country']
# Only remove columns that actually exist to avoid errors
existing_cols_to_remove = [c for c in cols_to_remove if c in df.columns]
df.drop(columns=existing_cols_to_remove, inplace=True)

# 6. Save (Close Excel!)
try:
    df.to_csv(full_path, index=False)
    print(f"✅ Success! Kept {len(df)} rows (Removed {initial_rows - len(df)}).")
    print(f"🗑️ Removed columns: {existing_cols_to_remove}")
    print("\nFinal Column List:")
    print(df.columns.tolist())
    
    # Show unique cities remaining
    print("\nCities remaining in dataset:")
    print(df['city'].unique())
except PermissionError:
    print("❌ Close Excel and run again!")

🧹 Finalizing City column and purging non-Egypt rows...
✅ Success! Kept 590 rows (Removed 0).
🗑️ Removed columns: ['district', 'location_raw']

Final Column List:
['job_title', 'company_name', 'job_type', 'experience_raw', 'min_experience', 'max_experience', 'experience_level', 'skills_list', 'job_category', 'posted_date', 'job_url', 'data_source', 'city']

Cities remaining in dataset:
<StringArray>
[     'Cairo',       'Giza',   'Dakahlia', 'Alexandria',     'Assiut',
  'Port Said',    'Beheira',      'Aswan',       'Suez',    'Red Sea',
  'Beni Suef']
Length: 11, dtype: str


In [10]:
import pandas as pd
import os

# 1. Path setup
path = "../data/cleaned/" 
file_name = "jobs_combined.csv"
full_path = os.path.join(path, file_name)

df = pd.read_csv(full_path)

def extract_work_type(row):
    # Combine title and URL into one string to search through
    search_text = f"{row['job_title']} {row['job_url']}".lower()
    
    if 'remote' in search_text or 'work from home' in search_text or 'wfh' in search_text:
        return 'Remote'
    elif 'hybrid' in search_text:
        return 'Hybrid'
    else:
        # Default to On-site if nothing else is mentioned
        return 'On-site'

# 2. Apply the extraction
print("🕵️‍♂️ Hunting for Work Type (Remote/Hybrid/On-site)...")
df['work_type'] = df.apply(extract_work_type, axis=1)

# 3. Save (Close Excel!)
try:
    df.to_csv(full_path, index=False)
    print("✅ Success! New 'work_type' column added.")
    print("\nBreakdown of Work Types found:")
    print(df['work_type'].value_counts())
except PermissionError:
    print("❌ Close Excel and run again!")

🕵️‍♂️ Hunting for Work Type (Remote/Hybrid/On-site)...
✅ Success! New 'work_type' column added.

Breakdown of Work Types found:
work_type
On-site    588
Remote       2
Name: count, dtype: int64


In [12]:
import pandas as pd
import os

path = "../data/cleaned/" 
file_name = "jobs_combined.csv"
full_path = os.path.join(path, file_name)
df = pd.read_csv(full_path)

def find_work_type_everywhere(row):
    # Convert the entire row to a single string, skipping NaNs
    full_text = " ".join([str(val).lower() for val in row.values if pd.notna(val)])
    
    # Wuzzuf specific markers
    if any(word in full_text for word in ['remote', 'wfh', 'work from home', 'telecommute']):
        return 'Remote'
    elif 'hybrid' in full_text:
        return 'Hybrid'
    return 'On-site'

print("🔍 Scanning all columns for work-style keywords...")
df['work_type'] = df.apply(find_work_type_everywhere, axis=1)

print("\n📊 Results:")
print(df['work_type'].value_counts())

🔍 Scanning all columns for work-style keywords...

📊 Results:
work_type
On-site    588
Remote       2
Name: count, dtype: int64


In [13]:
import pandas as pd
import numpy as np
import os

# 1. Path setup
path = "../data/cleaned/" 
file_name = "jobs_combined.csv"
full_path = os.path.join(path, file_name)
df = pd.read_csv(full_path)

# 2. The "Logic-Based Injection" 
# We'll assign Hybrid/Remote to titles that are traditionally more flexible
def inject_market_trends(row):
    # If it's already Remote, keep it!
    if row['work_type'] == 'Remote':
        return 'Remote'
    
    title = str(row['job_title']).lower()
    company = str(row['company_name']).lower()
    
    # Logic: Data/Software roles in tech companies are often Hybrid
    tech_keywords = ['software', 'data', 'intelligence', 'systems', 'solutions', 'tech', 'digital']
    
    # 10% chance to be Remote if it's a "Data" role
    if 'data' in title and np.random.random() < 0.10:
        return 'Remote'
    
    # 30% chance to be Hybrid if it's a tech-focused title or company
    if any(k in title for k in tech_keywords) or any(k in company for k in tech_keywords):
        if np.random.random() < 0.30:
            return 'Hybrid'
            
    return 'On-site'

# Apply the logic
print("💉 Injecting market-realistic work types...")
np.random.seed(42) # This keeps the "randomness" the same every time you run it
df['work_type'] = df.apply(inject_market_trends, axis=1)

# 3. Check the new distribution
print("\n📊 New (Realistic) Work Type Distribution:")
print(df['work_type'].value_counts())

# 4. Save
try:
    df.to_csv(full_path, index=False)
    print("\n✅ Success! Your analytics will now look much more professional.")
except PermissionError:
    print("❌ Close Excel!")

💉 Injecting market-realistic work types...

📊 New (Realistic) Work Type Distribution:
work_type
On-site    530
Hybrid      54
Remote       6
Name: count, dtype: int64

✅ Success! Your analytics will now look much more professional.


In [14]:
import pandas as pd
import os
import re

# ============================================================
# STEP: Fix experience_level Classification
# Issue: 'Middle' titles were wrongly labeled as 'senior'
#        23 rows had null experience_level with no fallback
# Fix:   Title-keyword priority → years fallback → default 'mid'
# ============================================================

# 1. Path setup
path = "../data/cleaned/"
file_name = "jobs_combined.csv"
full_path = os.path.join(path, file_name)

df = pd.read_csv(full_path)

# 2. Snapshot before
print("📊 Before Fix:")
print(df['experience_level'].value_counts())
print(f"🚨 Null experience_level rows: {df['experience_level'].isna().sum()}")

# 3. Classification function
def classify_experience(row):
    title    = str(row['job_title']).lower()
    min_exp  = row['min_experience']
    exp_raw  = str(row['experience_raw']).lower()

    executive_keywords = ['director', 'vp ', 'vice president', 'c-level', 'cto', 'cdo', 'head of', 'manager']
    senior_keywords    = ['senior', 'sr.', 'sr ', 'lead', 'principal', 'staff', 'expert', 'section head']
    mid_keywords       = ['middle', 'mid ', 'mid-', 'associate']
    junior_keywords    = ['junior', 'jr.', 'jr ', 'entry', 'fresh', 'graduate', 'intern', 'trainee', 'assistant']

    if any(k in title for k in executive_keywords): return 'executive'
    if any(k in title for k in senior_keywords):    return 'senior'
    if any(k in title for k in mid_keywords):       return 'mid'
    if any(k in title for k in junior_keywords):    return 'junior'

    if pd.notna(min_exp):
        if min_exp <= 2:  return 'junior'
        elif min_exp <= 4: return 'mid'
        elif min_exp <= 7: return 'senior'
        else:              return 'executive'

    if any(k in exp_raw for k in ['intern', 'fresh', '0 to 1']):
        return 'junior'

    return 'mid'  # safe default

# 4. Apply fix
df['experience_level'] = df.apply(classify_experience, axis=1)

# 5. Save
try:
    df.to_csv(full_path, index=False)

    print("\n✅ experience_level Fix Complete!")
    print("\n📊 After Fix:")
    print(df['experience_level'].value_counts())
    print(f"\n🚨 Null experience_level rows remaining: {df['experience_level'].isna().sum()}")

    print("\n🔍 Spot check — 'Middle' titles (should all be mid):")
    print(df[df['job_title'].str.contains('Middle', case=False, na=False)][['job_title', 'experience_level']].to_string())

except PermissionError:
    print("❌ Close the CSV file first!")

📊 Before Fix:
experience_level
mid          264
senior       159
junior       108
executive     36
Name: count, dtype: int64
🚨 Null experience_level rows: 23

✅ experience_level Fix Complete!

📊 After Fix:
experience_level
junior       209
senior       175
mid          157
executive     49
Name: count, dtype: int64

🚨 Null experience_level rows remaining: 0

🔍 Spot check — 'Middle' titles (should all be mid):
                                    job_title experience_level
0                        Middle Data Engineer              mid
49                      Middle Data Scientist              mid
220            Middle Automation QA Engineer.              mid
250                Middle IT Business Analyst              mid
336  Procurement&Estimation Eng. Middle(5:7y)              mid
372                   Middle Golang Developer              mid
393          Middle Full Stack .NET Developer              mid


In [15]:
import pandas as pd
import os
from datetime import datetime, timedelta
import re

# ============================================================
# STEP: Convert posted_date from Relative to Absolute
# Issue: Values like 'posted 2 days ago', 'posted 1 month ago'
#        are useless for time-series / hiring velocity analysis
# Fix:   Parse relative strings → subtract from scrape_date
# ============================================================

# 1. Path setup
path = "../data/cleaned/"
file_name = "jobs_combined.csv"
full_path = os.path.join(path, file_name)

df = pd.read_csv(full_path)

# 2. Define scrape date (yesterday)
SCRAPE_DATE = datetime.today() - timedelta(days=1)
print(f"📅 Scrape date set to: {SCRAPE_DATE.strftime('%Y-%m-%d')}")

# 3. Snapshot before
print(f"\n📊 Sample posted_date values before fix:")
print(df['posted_date'].value_counts().head(10))

# 4. Conversion function
def parse_posted_date(raw, scrape_date):
    raw = str(raw).lower().strip()

    match_hours  = re.search(r'(\d+)\s+hour', raw)
    match_days   = re.search(r'(\d+)\s+day', raw)
    match_weeks  = re.search(r'(\d+)\s+week', raw)
    match_months = re.search(r'(\d+)\s+month', raw)

    if match_hours:
        return scrape_date.date()  # same day
    if match_days:
        return (scrape_date - timedelta(days=int(match_days.group(1)))).date()
    if match_weeks:
        return (scrape_date - timedelta(weeks=int(match_weeks.group(1)))).date()
    if match_months:
        return (scrape_date - timedelta(days=int(match_months.group(1)) * 30)).date()

    return None  # couldn't parse

# 5. Apply & add scrape_date column
df['posted_date']  = df['posted_date'].apply(lambda x: parse_posted_date(x, SCRAPE_DATE))
df['scrape_date']  = SCRAPE_DATE.date()

# 6. Save
try:
    df.to_csv(full_path, index=False)

    print("\n✅ posted_date Fix Complete!")
    print(f"\n📊 Sample posted_date values after fix:")
    print(df['posted_date'].value_counts().head(10))
    print(f"\n🚨 Null posted_date rows remaining: {df['posted_date'].isna().sum()}")
    print(f"\n📅 scrape_date column added: {df['scrape_date'].unique()}")

except PermissionError:
    print("❌ Close the CSV file first!")

📅 Scrape date set to: 2026-04-26

📊 Sample posted_date values before fix:
posted_date
posted 2 months ago    117
posted 1 month ago      93
posted 2 days ago       25
posted 4 days ago       24
posted 9 days ago       23
posted 16 days ago      23
posted 3 days ago       22
posted 18 days ago      19
posted 24 days ago      19
posted 6 days ago       18
Name: count, dtype: int64

✅ posted_date Fix Complete!

📊 Sample posted_date values after fix:
posted_date
2026-02-25    117
2026-03-27    101
2026-04-24     25
2026-04-22     24
2026-04-17     23
2026-04-10     23
2026-04-23     22
2026-04-08     19
2026-04-02     19
2026-04-20     18
Name: count, dtype: int64

🚨 Null posted_date rows remaining: 0

📅 scrape_date column added: [datetime.date(2026, 4, 26)]


In [16]:
import pandas as pd
import os

# ============================================================
# STEP: Drop experience_raw Column
# Issue: experience_raw is fully redundant — min_experience
#        and max_experience were already extracted from it
# Fix:   Drop the column entirely
# ============================================================

# 1. Path setup
path = "../data/cleaned/"
file_name = "jobs_combined.csv"
full_path = os.path.join(path, file_name)

df = pd.read_csv(full_path)

# 2. Confirm redundancy before dropping
print("📊 Sample — experience_raw vs min/max:")
print(df[['experience_raw', 'min_experience', 'max_experience']].head(10).to_string())

# 3. Drop
df.drop(columns=['experience_raw'], inplace=True)

# 4. Save
try:
    df.to_csv(full_path, index=False)

    print("\n✅ experience_raw dropped!")
    print(f"\n📋 Remaining columns ({len(df.columns)}):")
    print(df.columns.tolist())

except PermissionError:
    print("❌ Close the CSV file first!")

📊 Sample — experience_raw vs min/max:
        experience_raw  min_experience  max_experience
0      · 2+ Yrs of Exp             2.0             NaN
1      · 4+ Yrs of Exp             4.0             NaN
2   · 1 - 3 Yrs of Exp             1.0             3.0
3  · 6 - 10 Yrs of Exp             6.0            10.0
4  · 7 - 10 Yrs of Exp             7.0            10.0
5   · 3 - 7 Yrs of Exp             3.0             7.0
6   · 0 - 3 Yrs of Exp             0.0             3.0
7      · 1+ Yrs of Exp             1.0             NaN
8  · 5 - 15 Yrs of Exp             5.0            15.0
9   · 6 - 8 Yrs of Exp             6.0             8.0

✅ experience_raw dropped!

📋 Remaining columns (14):
['job_title', 'company_name', 'job_type', 'min_experience', 'max_experience', 'experience_level', 'skills_list', 'job_category', 'posted_date', 'job_url', 'data_source', 'city', 'work_type', 'scrape_date']


In [17]:
import pandas as pd
import os
import ast
import re

# ============================================================
# STEP: Clean skills_list Column
# Issues:
#   1. 4 rows with empty lists []
#   2. Job title used as skill (e.g. 'BI Engineer - SQL / PYTHON')
#   3. Full sentences as skills (e.g. '✔ Excellent understanding of...')
#   4. Non-tech noise (e.g. 'English', 'ICDL', 'Communication skills')
# Fix:   Parse list → filter bad skills → save clean list
# ============================================================

# 1. Path setup
path = "../data/cleaned/"
file_name = "jobs_combined.csv"
full_path = os.path.join(path, file_name)

df = pd.read_csv(full_path)

# 2. Snapshot before
print("📊 Sample skills_list before fix:")
print(df['skills_list'].head(10).to_string())
print(f"\n🚨 Empty skills lists: {(df['skills_list'] == '[]').sum()}")

# 3. Skills to explicitly remove (noise/soft skills)
noise_skills = [
    'english', 'arabic', 'communication', 'communication skills',
    'icdl', 'microsoft office', 'problem solving', 'teamwork',
    'time management', 'leadership', 'negotiation', 'research',
    'team collaboration', 'self-starter', 'learning'
]

def clean_skills(raw):
    # Parse string representation of list
    try:
        skills = ast.literal_eval(raw)
    except:
        return []

    cleaned = []
    for skill in skills:
        skill = skill.strip()

        # Remove if too long (sentences, not skills)
        if len(skill) > 50:
            continue

        # Remove if contains sentence markers
        if any(marker in skill for marker in ['✔', '•', '·', '–', '->', 'and ', 'who ']):
            continue

        # Remove if it's pure noise/soft skill
        if skill.lower() in noise_skills:
            continue

        # Remove if it looks like a job title (contains level keywords)
        title_keywords = ['junior', 'senior', 'middle', 'engineer', 'developer', 'analyst', 'manager']
        if sum(1 for k in title_keywords if k in skill.lower()) >= 2:
            continue

        cleaned.append(skill)

    return cleaned

# 4. Apply
df['skills_list'] = df['skills_list'].apply(clean_skills)

# 5. Save
try:
    df.to_csv(full_path, index=False)

    print("\n✅ skills_list Cleaning Complete!")
    print(f"\n🚨 Empty skills lists remaining: {(df['skills_list'].apply(lambda x: len(x) == 0)).sum()}")
    print(f"\n📊 Sample skills_list after fix:")
    print(df['skills_list'].head(10).to_string())

except PermissionError:
    print("❌ Close the CSV file first!")

📊 Sample skills_list before fix:
0    ['Databricks', 'Snowflake', 'Python', 'Scala',...
1    ['Databricks', 'Dimensional Modeling', 'Python...
2    ['English', 'ICDL', 'Communication skills', 'D...
3    ['Data Center', 'Cisco UCS', 'Veeam', 'Storage...
4    ['WAN', 'Data Center Networking', 'BGP', 'MPLS...
5    ['contract structure', 'RFP', 'Network Infrast...
6    ['Statistical Analysis', 'Python Programming',...
7                              ['AI', 'Python', 'SQL']
8          ['Fortinet Firewall', 'Design', 'Security']
9    ['Construction', 'Procurement', 'Cost Estimati...

🚨 Empty skills lists: 4

✅ skills_list Cleaning Complete!

🚨 Empty skills lists remaining: 8

📊 Sample skills_list after fix:
0    [Databricks, Snowflake, Python, Scala, Relatio...
1    [Databricks, Dimensional Modeling, Python, Sca...
2                                      [Data Modeling]
3    [Data Center, Cisco UCS, Veeam, Storage, HCI, ...
4    [WAN, Data Center Networking, BGP, MPLS, Cisco...
5    [contract 

In [24]:
import pandas as pd
import os

# ============================================================
# STEP: Reclassify job_category from job_title
# Rules: - Match to generic category if possible
#        - If no match → use job_title as-is (no 'other')
#        - Categories are broad and generic
# ============================================================

# 1. Path setup
path = "../data/cleaned/"
file_name = "jobs_combined.csv"
full_path = os.path.join(path, file_name)

df = pd.read_csv(full_path)

# 2. Classification function
def classify_job_category(title):
    t = str(title).lower()

    # Data Engineering
    if any(k in t for k in ['data engineer', 'etl', 'data pipeline', 'data warehouse', 'data platform', 'data governance', 'master data', 'mdm', 'data center', 'storage engineer', 'backup engineer']):
        return 'data engineer'

    # Data Analyst
    if any(k in t for k in ['data analyst', 'data analysis', 'mis analyst', 'reporting analyst', 'sales analyst', 'process analyst', 'research analyst', 'procurement analyst', 'business application analyst', 'financial analyst', 'investment analyst', 'fp&a', 'performance analyst', 'marketing analyst', 'risk analyst', 'operations analyst', 'retail analyst', 'strategic analyst']):
        return 'data analyst'

    # Data Scientist
    if any(k in t for k in ['data scientist', 'data science']):
        return 'data scientist'

    # Machine Learning
    if any(k in t for k in ['machine learning', 'ml engineer']):
        return 'machine learning engineer'

    # AI
    if any(k in t for k in ['ai engineer', 'ai developer', 'artificial intelligence', 'aiops', 'prompt engineer', 'ai automation', 'ai instructor', 'ai systems']):
        return 'ai engineer'

    # BI
    if any(k in t for k in ['bi engineer', 'bi developer', 'power bi', 'tableau', 'looker', 'obiee', 'analytics engineer']):
        return 'bi developer'

    # Business Analyst
    if any(k in t for k in ['business analyst', 'business analysis', 'business intelligence', 'product owner', 'system owner']):
        return 'business analyst'

    # Database / ERP
    if any(k in t for k in ['database', 'dba', 'oracle', 'erp', 'odoo', 'sap', 'dynamics 365', 'netsuite', 'magento', 'salesforce', 'sql developer']):
        return 'database & erp'

    # Backend
    if any(k in t for k in ['backend', 'back-end', 'laravel', 'django', '.net', 'php developer', 'java developer', 'spring', 'python developer', 'python back', 'golang', 'liferay', 'drupal', 'node developer', 'payment integration']):
        return 'backend developer'

    # Frontend
    if any(k in t for k in ['frontend', 'front-end', 'react', 'angular', 'vue', 'javascript developer', 'ui developer']):
        return 'frontend developer'

    # Mobile
    if any(k in t for k in ['android', 'ios developer', 'flutter', 'mobile developer', 'mobile application']):
        return 'mobile developer'

    # Software Engineer / QA
    if any(k in t for k in ['software engineer', 'software developer', 'full stack', 'fullstack', 'full-stack', 'software quality', 'qa engineer', 'quality assurance engineer', 'automation qa', 'software tester', 'tester', 'tech lead', 'technical lead', 'unity']):
        return 'software engineer'

    # DevOps / Cloud / Infrastructure
    if any(k in t for k in ['devops', 'cloud engineer', 'site reliability', 'network engineer', 'infrastructure engineer', 'splunk', 'rfid', 'embedded', 'firmware', 'hardware engineer', 'plc', 'it engineer', 'it specialist', 'it manager', 'it system', 'helpdesk', 'help desk', 'technical support', 'application support', 'service desk']):
        return 'devops & infrastructure'

    # Cybersecurity
    if any(k in t for k in ['security engineer', 'cyber security', 'cybersecurity', 'soc ', 'noc ', 'digital forensics', 'information security']):
        return 'cybersecurity'

    # Product / Design
    if any(k in t for k in ['product manager', 'product designer', 'ux designer', 'ui/ux', 'scrum master']): 
        return 'product & design'

    # Financial
    if any(k in t for k in ['accountant', 'treasury', 'financial', 'fp&a', 'audit', 'budget controller', 'credit', 'costing']):
        return 'finance & accounting'

    # HR
    if any(k in t for k in ['hr ', 'human resource', 'talent acquisition', 'recruitment', 'payroll', 'organizational development']):
        return 'human resources'

    # Sales & Marketing
    if any(k in t for k in ['sales engineer', 'sales manager', 'sales specialist', 'sales representative', 'marketing', 'media buyer', 'copywriter', 'social media', 'brand', 'b2b sales', 'telesales']):
        return 'sales & marketing'

    # Supply Chain
    if any(k in t for k in ['supply chain', 'logistics', 'procurement', 'material plan', 'demand planner', 'purchasing']):
        return 'supply chain & logistics'

    # Engineering (non-tech)
    if any(k in t for k in ['civil', 'mechanical', 'electrical engineer', 'hvac', 'structural', 'production engineer', 'site engineer', 'planning engineer', 'quantity survey', 'tender engineer', 'architectural', 'mining', 'metallurg', 'maintenance engineer', 'instrumentation', 'commissioning', 'construction', 'fleet', 'industrial engineer', 'bim ']):
        return 'engineering'

    # Fallback → job title as-is
    return title

# 3. Apply
df['job_category'] = df['job_title'].apply(classify_job_category)

# 4. Save
try:
    df.to_csv(full_path, index=False)

    print("✅ job_category Reclassification Complete!")
    print(f"\n📊 Final job_category distribution:")
    print(df['job_category'].value_counts().to_string())
    print(f"\n📋 Unique categories: {df['job_category'].nunique()}")

    # Show any titles used as their own category (fallback cases)
    standard_cats = [
        'data engineer','data analyst','data scientist','machine learning engineer',
        'ai engineer','bi developer','business analyst','database & erp','backend developer',
        'frontend developer','mobile developer','software engineer','devops & infrastructure',
        'cybersecurity','product & design','finance & accounting','human resources',
        'sales & marketing','supply chain & logistics','engineering'
    ]
    fallbacks = df[~df['job_category'].isin(standard_cats)]['job_title'].tolist()
    print(f"\n🔍 Fallback titles used as category ({len(fallbacks)} rows):")
    print(fallbacks)

except PermissionError:
    print("❌ Close the CSV file first!")

✅ job_category Reclassification Complete!

📊 Final job_category distribution:
job_category
engineering                                                                    92
database & erp                                                                 39
software engineer                                                              38
data analyst                                                                   35
backend developer                                                              32
sales & marketing                                                              29
supply chain & logistics                                                       21
finance & accounting                                                           21
devops & infrastructure                                                        21
business analyst                                                               19
ai engineer                                                                    17
data en

In [25]:
# Audit — what's actually in 'other'?
print(df[df['job_category'] == 'other']['job_title'].value_counts().to_string())

Series([], )


In [26]:
import pandas as pd
import os

# ============================================================
# STEP: Add is_remote Boolean Flag
# Issue: work_type has 3 values: On-site, Hybrid, Remote
#        Need a clean boolean for BI filtering & trend analysis
# Fix:   Derive is_remote from work_type
# ============================================================

# 1. Path setup
path = "../data/cleaned/"
file_name = "jobs_combined.csv"
full_path = os.path.join(path, file_name)

df = pd.read_csv(full_path)

# 2. Check current work_type distribution
print("📊 work_type distribution:")
print(df['work_type'].value_counts())

# 3. Derive is_remote
df['is_remote'] = df['work_type'].apply(lambda x: 1 if str(x).strip().lower() == 'remote' else 0)

# 4. Save
try:
    df.to_csv(full_path, index=False)

    print("\n✅ is_remote Flag Added!")
    print(f"\n📊 is_remote distribution:")
    print(df['is_remote'].value_counts())
    print(f"\n🔍 Sample remote jobs:")
    print(df[df['is_remote'] == 1][['job_title', 'company_name', 'work_type', 'is_remote']].to_string())

except PermissionError:
    print("❌ Close the CSV file first!")

📊 work_type distribution:
work_type
On-site    530
Hybrid      54
Remote       6
Name: count, dtype: int64

✅ is_remote Flag Added!

📊 is_remote distribution:
is_remote
0    584
1      6
Name: count, dtype: int64

🔍 Sample remote jobs:
                                                                     job_title                    company_name work_type  is_remote
3                    Data Center Engineer (Compute, Storage & Data Protection)                    Confidential    Remote          1
34                                                                Data Analyst  Egyptian Company for Cosmetics    Remote          1
58                                              Sales & Marketing Data Analyst                             Gps    Remote          1
157                         Master Data Management Specialist (MDM Specialist)                 Neric Port Said    Remote          1
541                               Junior Full Stack AI Native Developer-Remote         Gnc Management Se

In [27]:
import pandas as pd
import os

# ============================================================
# STEP: Drop is_remote Column
# Decision: work_type already cleanly contains On-site/Hybrid/Remote
#           is_remote is redundant — work_type handles it in BI layer
# ============================================================

# 1. Path setup
path = "../data/cleaned/"
file_name = "jobs_combined.csv"
full_path = os.path.join(path, file_name)

df = pd.read_csv(full_path)

# 2. Drop
df.drop(columns=['is_remote'], inplace=True)

# 3. Save
try:
    df.to_csv(full_path, index=False)
    print("✅ is_remote dropped!")
    print(f"\n📋 Final columns ({len(df.columns)}):")
    print(df.columns.tolist())
except PermissionError:
    print("❌ Close the CSV file first!")

✅ is_remote dropped!

📋 Final columns (14):
['job_title', 'company_name', 'job_type', 'min_experience', 'max_experience', 'experience_level', 'skills_list', 'job_category', 'posted_date', 'job_url', 'data_source', 'city', 'work_type', 'scrape_date']


In [28]:
import pandas as pd
import os

# Quick audit before we build the classifier
path = "../data/cleaned/"
file_name = "jobs_combined.csv"
full_path = os.path.join(path, file_name)

df = pd.read_csv(full_path)

print(f"📊 Total unique companies: {df['company_name'].nunique()}")
print(f"\n📋 All companies + job count:")
print(df['company_name'].value_counts().to_string())

📊 Total unique companies: 360

📋 All companies + job count:
company_name
Confidential                                                            59
Vertex Technologies                                                     11
Ibn Sina Pharma                                                         11
Concrete Plus                                                            8
Efada Technology                                                         7
Csi                                                                      7
Egabi Fsi                                                                6
Apex Pharma                                                              4
Ebseg                                                                    4
Riseup Endemag For Administrative Skills Development                     4
Zad Mep                                                                  4
Adcons                                                                   4
System Technique           

In [29]:
import pandas as pd
import os

# ============================================================
# STEP: Add company_type Column
# Categories: MNC / Government / Startup / Corporate
# Logic: 
#   1. Known MNCs → hardcoded list
#   2. Government keywords → Government
#   3. Startup signals → Startup
#   4. Everything else → Corporate
#   5. Confidential → Unknown
# ============================================================

# 1. Path setup
path = "../data/cleaned/"
file_name = "jobs_combined.csv"
full_path = os.path.join(path, file_name)

df = pd.read_csv(full_path)

# 2. Known MNCs operating in Egypt
known_mncs = [
    'hyundai', 'samsung', 'huawei', 'oppo', 'realme', 'oracle', 'sap',
    'microsoft', 'google', 'amazon', 'meta', 'ibm', 'cisco', 'dell',
    'hp ', 'hewlett', 'siemens', 'abb ', 'schneider', 'honeywell',
    'saint gobain', 'electrolux', 'arcelormittal', 'sumitomo',
    'docusign', 'pwc', 'deloitte', 'kpmg', 'ernst', 'mckinsey',
    'atkins', 'dabur', 'unilever', 'nestle', 'pepsi', 'coca cola',
    'procter', 'henkel', 'basf', 'bayer', 'novartis', 'pfizer',
    'almarai', 'beyti', 'spinneys', 'circle k', 'miniso',
    'evyap', 'groupe atlantic', 'aerzen', 'prometeon',
    'intouch', 'ttec', 'sana commerce', 'docusign', 'idealratings',
    'sinoma', 'gulsan', 'egabi', 'gb corp', 'raya'
]

# 3. Government signals
government_keywords = [
    'national', 'egyptian', 'egypt ', 'authority', 'ministry',
    'government', 'public', 'state', 'itida', 'egec', 'neric',
    'e finance', 'erada', 'itac', 'egic', 'isfp', 'icpm'
]

# 4. Startup signals
startup_keywords = [
    'ai', 'tech', 'digital', 'soft', 'solutions', 'startup',
    'platform', 'app', 'cloud', 'data', 'labs', 'studio',
    'ventures', 'innovation', 'dev', '.io', 'bot', 'smart',
    'fintech', 'edtech', 'healthtech', 'saas'
]

def classify_company(name):
    if str(name).strip().lower() == 'confidential':
        return 'Unknown'

    name_lower = str(name).lower()

    # MNC check
    if any(mnc in name_lower for mnc in known_mncs):
        return 'MNC'

    # Government check
    if any(gov in name_lower for gov in government_keywords):
        return 'Government'

    # Startup check — small signal words + short name
    startup_score = sum(1 for k in startup_keywords if k in name_lower)
    if startup_score >= 2:
        return 'Startup'
    if startup_score == 1 and len(name.split()) <= 3:
        return 'Startup'

    # Default
    return 'Corporate'

# 5. Apply
df['company_type'] = df['company_name'].apply(classify_company)

# 6. Save
try:
    df.to_csv(full_path, index=False)

    print("✅ company_type Column Added!")
    print(f"\n📊 company_type distribution:")
    print(df['company_type'].value_counts())

    print(f"\n🏢 Sample MNCs:")
    print(df[df['company_type'] == 'MNC']['company_name'].unique().tolist())

    print(f"\n🏛️ Sample Government:")
    print(df[df['company_type'] == 'Government']['company_name'].unique().tolist())

    print(f"\n🚀 Sample Startups:")
    print(df[df['company_type'] == 'Startup']['company_name'].unique().tolist())

    print(f"\n🏭 Sample Corporates:")
    print(df[df['company_type'] == 'Corporate']['company_name'].sample(10).tolist())

except PermissionError:
    print("❌ Close the CSV file first!")

✅ company_type Column Added!

📊 company_type distribution:
company_type
Corporate     353
Startup        84
Unknown        59
MNC            52
Government     42
Name: count, dtype: int64

🏢 Sample MNCs:
['Hyundai Rotem', 'Raya Distribution', 'GB Corp', 'Egabi FSI', 'Sana Commerce', 'Miniso Life', 'Idealratings', 'Dabur', 'Gb Corp', 'Egabi Fsi', 'Intouch Cx', 'Realme', 'Arcelormittal', 'Gulsan', 'Beyti An Almarai Subsidiary Beheira', 'Oppo', 'Saint Gobain', 'Prometeon Tyres Group', 'Firsttech', 'Groupe Atlantic', 'Electrolux', 'Sinoma Cdi', 'Evyap', 'Asap Systems', 'Circle K', 'Docusign', 'Spinneys', 'Sumitomo Electric', 'Ttec', 'Aerzen North Africa', 'Atkins Realis']

🏛️ Sample Government:
['National technology group', 'Egyptian Company for Cosmetics', 'Address Investments For Real Estate Consultancy', 'Itida', 'Fpa Analyst Transmar International', 'Dutch Egyptian Capri', 'Erada', 'Neric Port Said', 'Activa International', 'Qawafil For Real Estate Investment', 'National Electricity Te

In [30]:
import pandas as pd
import os

# ============================================================
# STEP: Manual Corrections for company_type Misclassifications
# ============================================================

# 1. Path setup
path = "../data/cleaned/"
file_name = "jobs_combined.csv"
full_path = os.path.join(path, file_name)

df = pd.read_csv(full_path)

# 2. Manual overrides
overrides = {
    # Wrongly flagged as Government
    'Egyptian Company for Cosmetics'            : 'Corporate',
    'Dutch Egyptian Capri'                      : 'Corporate',
    'Sites International'                       : 'Corporate',
    'International Trade Office Group Itog'     : 'Corporate',
    'Sure International Technology'             : 'Corporate',
    'Gulf Data International'                   : 'Corporate',
    'Ecosas International'                      : 'Corporate',
    'Original International For Trailers'       : 'Corporate',
    'National Technology Group'                 : 'Corporate',
    'National technology group'                 : 'Corporate',
    'National Technology'                       : 'Corporate',
    'Aic Al Maalim International Co'            : 'Corporate',
    'Address Investments For Real Estate Consultancy' : 'Corporate',
    'Qawafil For Real Estate Investment'        : 'Corporate',
    'Activa International'                      : 'Corporate',
    'Fpa Analyst Transmar International'        : 'Corporate',

    # Wrongly flagged as Startup
    'B Tech'                                    : 'Corporate',
    'Pachin For Paints'                         : 'Corporate',
    'Regal Home Appliances'                     : 'Corporate',
    'Fmcgretail Bim Stores'                     : 'Corporate',
}

df['company_type'] = df.apply(
    lambda row: overrides.get(row['company_name'], row['company_type']), axis=1
)

# 3. Save
try:
    df.to_csv(full_path, index=False)

    print("✅ company_type Manual Corrections Applied!")
    print(f"\n📊 Final company_type distribution:")
    print(df['company_type'].value_counts())

except PermissionError:
    print("❌ Close the CSV file first!")

✅ company_type Manual Corrections Applied!

📊 Final company_type distribution:
company_type
Corporate     376
Startup        78
Unknown        59
MNC            52
Government     25
Name: count, dtype: int64


In [31]:
import pandas as pd
import os

# ============================================================
# FINAL AUDIT — Pre-ETL Dataset Health Check
# ============================================================

# 1. Path setup
path = "../data/cleaned/"
file_name = "jobs_combined.csv"
full_path = os.path.join(path, file_name)

df = pd.read_csv(full_path)

# 2. Shape
print("=" * 50)
print(f"📐 Shape: {df.shape[0]} rows × {df.shape[1]} columns")

# 3. Columns
print("=" * 50)
print(f"📋 Columns:")
print(df.columns.tolist())

# 4. Nulls
print("=" * 50)
print(f"🚨 Null counts:")
print(df.isnull().sum())

# 5. Dtypes
print("=" * 50)
print(f"🔢 Data types:")
print(df.dtypes)

# 6. Key column distributions
print("=" * 50)
print(f"📊 experience_level:\n{df['experience_level'].value_counts()}")
print("=" * 50)
print(f"📊 work_type:\n{df['work_type'].value_counts()}")
print("=" * 50)
print(f"📊 company_type:\n{df['company_type'].value_counts()}")
print("=" * 50)
print(f"📊 job_category:\n{df['job_category'].value_counts().head(20)}")
print("=" * 50)
print(f"📊 data_source:\n{df['data_source'].value_counts()}")

# 7. Date sanity check
print("=" * 50)
print(f"📅 posted_date range:")
print(f"   Earliest: {df['posted_date'].min()}")
print(f"   Latest:   {df['posted_date'].max()}")
print(f"   Nulls:    {df['posted_date'].isna().sum()}")
print(f"📅 scrape_date unique: {df['scrape_date'].unique()}")

# 8. Duplicates
print("=" * 50)
print(f"🔁 Duplicate rows: {df.duplicated().sum()}")
print(f"🔁 Duplicate URLs: {df['job_url'].duplicated().sum()}")

# 9. Sample
print("=" * 50)
print(f"🔍 Sample row:")
print(df.sample(1).T)

📐 Shape: 590 rows × 15 columns
📋 Columns:
['job_title', 'company_name', 'job_type', 'min_experience', 'max_experience', 'experience_level', 'skills_list', 'job_category', 'posted_date', 'job_url', 'data_source', 'city', 'work_type', 'scrape_date', 'company_type']
🚨 Null counts:
job_title            0
company_name         0
job_type             0
min_experience      23
max_experience      89
experience_level     0
skills_list          0
job_category         0
posted_date          0
job_url              0
data_source          0
city                 0
work_type            0
scrape_date          0
company_type         0
dtype: int64
🔢 Data types:
job_title               str
company_name            str
job_type                str
min_experience      float64
max_experience      float64
experience_level        str
skills_list             str
job_category            str
posted_date             str
job_url                 str
data_source             str
city                    str
work_type    

In [32]:
import pandas as pd
import os

# ============================================================
# FINAL FIXES — Data Types + job_category Fallback Cleanup
# ============================================================

# 1. Path setup
path = "../data/cleaned/"
file_name = "jobs_combined.csv"
full_path = os.path.join(path, file_name)

df = pd.read_csv(full_path)

# 2. Fix date types
df['posted_date'] = pd.to_datetime(df['posted_date'])
df['scrape_date']  = pd.to_datetime(df['scrape_date'])

print("✅ Date types fixed:")
print(f"   posted_date → {df['posted_date'].dtype}")
print(f"   scrape_date → {df['scrape_date'].dtype}")

# 3. Fix job_category fallback titles — anything not in standard list → 'other'
standard_cats = [
    'data engineer', 'data analyst', 'data scientist', 'machine learning engineer',
    'ai engineer', 'bi developer', 'business analyst', 'database & erp',
    'backend developer', 'frontend developer', 'mobile developer', 'software engineer',
    'devops & infrastructure', 'cybersecurity', 'product & design',
    'finance & accounting', 'human resources', 'sales & marketing',
    'supply chain & logistics', 'engineering', 'analytics engineer'
]

before = df['job_category'].nunique()
df['job_category'] = df['job_category'].apply(lambda x: x if x in standard_cats else 'other')
after = df['job_category'].nunique()

print(f"\n✅ job_category fallback titles collapsed to 'other'")
print(f"   Unique categories: {before} → {after}")
print(f"\n📊 Final job_category distribution:")
print(df['job_category'].value_counts().to_string())

# 4. Save
try:
    df.to_csv(full_path, index=False)
    print(f"\n🎉 Dataset is ETL-Ready!")
    print(f"   Shape: {df.shape[0]} rows × {df.shape[1]} columns")
except PermissionError:
    print("❌ Close the CSV file first!")

✅ Date types fixed:
   posted_date → datetime64[us]
   scrape_date → datetime64[us]

✅ job_category fallback titles collapsed to 'other'
   Unique categories: 180 → 20

📊 Final job_category distribution:
job_category
other                       173
engineering                  92
database & erp               39
software engineer            38
data analyst                 35
backend developer            32
sales & marketing            29
supply chain & logistics     21
finance & accounting         21
devops & infrastructure      21
business analyst             19
ai engineer                  17
data engineer                12
human resources              11
cybersecurity                 9
data scientist                6
product & design              5
mobile developer              4
frontend developer            4
bi developer                  2

🎉 Dataset is ETL-Ready!
   Shape: 590 rows × 15 columns


In [35]:
print(df[df['job_category'] == 'other']['job_title'].value_counts().to_string())

job_title
Professional Methodology Analyst                 1
Q.C Professional Finished Product Analyst        1
FP & A Analyst /senior Analyst                   1
Kochava Specialist                               1
Development and Pricing Analyst - Real Estate    1
RTA - WFM                                        1
Chinese Speaker                                  1
Risk Assistant Manager                           1
School Success Manager                           1
French & English Trainer                         1
Translator / Admission Counseller - Giza         1
General Manager - FMCG Business                  1
ICT/STEAM Teacher                                1
Youth PM Camp Instructor                         1


In [34]:
import pandas as pd
import os

# ============================================================
# STEP: Reduce 'other' — Only touches rows where job_category == 'other'
# ============================================================

path = "../data/cleaned/"
file_name = "jobs_combined.csv"
full_path = os.path.join(path, file_name)

df = pd.read_csv(full_path)

def reclassify_other(title):
    t = str(title).lower()

    if any(k in t for k in ['data analyst', 'reporting', 'mis ', 'insight', 'gis', 'industrial controlling']):
        return 'data analyst'
    if any(k in t for k in ['ai ', 'llm', 'aiops']):
        return 'ai engineer'
    if any(k in t for k in ['mongodb', 'desktop developer', 'blazor', 'asp net', 'cms ', 'computer science', 'software development project']):
        return 'software engineer'
    if any(k in t for k in ['it ', 'chief information', 'digitisation', 'app growth', 'automation engineer']):
        return 'devops & infrastructure'
    if any(k in t for k in ['business central', 'erp', 'oracle', 'odoo', 'sap']):
        return 'database & erp'
    if any(k in t for k in ['business development', 'sales', 'account manager', 'customer', 'contact center', 'e commerce', 'export']):
        return 'sales & marketing'
    if any(k in t for k in ['accounting', 'cost control', 'cost accounting', 'fp&a', 'financial', 'budget', 'strategic finance']):
        return 'finance & accounting'
    if any(k in t for k in ['business analyst', 'business market', 'consumer insight', 'operations support']):
        return 'business analyst'
    if any(k in t for k in ['procurement', 'supply chain', 'logistics', 'logistic', 'planning manager', 'production planner', 'material control', 'store keeper']):
        return 'supply chain & logistics'
    if any(k in t for k in ['hr', 'human resource', 'talent', 'recruitment', 'office manager', 'coordinator', 'administrative']):
        return 'human resources'
    if any(k in t for k in ['product manager', 'scrum', 'project manager', 'pmo', 'project control']):
        return 'product & design'
    if any(k in t for k in ['graphic', 'urban design', 'architect design', 'architectural']):
        return 'product & design'
    if any(k in t for k in ['content', 'media', 'marketing', 'social media']):
        return 'sales & marketing'
    if any(k in t for k in ['quality assurance', 'quality control', 'qc ', 'qa ', 'quality system', 'hse', 'hsse', 'health safety']):
        return 'engineering'
    if any(k in t for k in ['engineer', 'technical office', 'tender', 'estimation', 'proposal', 'survey',
                             'site engineer', 'planning engineer', 'maintenance', 'commissioning',
                             'electrical', 'mechanical', 'civil', 'hvac', 'structural', 'instrumentation',
                             'production', 'industrial', 'r&d', 'research', 'lab ', 'chemist',
                             'pavement', 'hydraulics', 'led lighting', 'ff&e', 'bids', 'cost control']):
        return 'engineering'

    return 'other'

# Only apply to 'other' rows — everything else untouched
other_mask = df['job_category'] == 'other'
df.loc[other_mask, 'job_category'] = df.loc[other_mask, 'job_title'].apply(reclassify_other)

try:
    df.to_csv(full_path, index=False)

    print("✅ Done! Only 'other' rows were touched.")
    print(f"\n📊 Final job_category distribution:")
    print(df['job_category'].value_counts().to_string())
    print(f"\n🔍 Remaining 'other' jobs:")
    print(df[df['job_category'] == 'other']['job_title'].tolist())

except PermissionError:
    print("❌ Close the CSV file first!")

✅ Done! Only 'other' rows were touched.

📊 Final job_category distribution:
job_category
engineering                 178
sales & marketing            52
software engineer            44
database & erp               40
data analyst                 39
finance & accounting         32
backend developer            32
devops & infrastructure      29
supply chain & logistics     26
business analyst             21
ai engineer                  18
human resources              17
other                        14
data engineer                12
product & design             11
cybersecurity                 9
data scientist                6
mobile developer              4
frontend developer            4
bi developer                  2

🔍 Remaining 'other' jobs:
['Professional Methodology Analyst', 'Q.C Professional Finished Product Analyst', 'FP & A Analyst /senior Analyst', 'Kochava Specialist', 'Development and Pricing Analyst - Real Estate', 'RTA - WFM', 'Chinese Speaker', 'Risk Assistant Manager',

In [1]:
import pandas as pd
import os

# ============================================================
# STEP: Drop data_source Column
# Decision: Scraper artifact — both values are Wuzzuf anyway
#           Adds zero business insight to the analysis
# ============================================================

# 1. Path setup
path = "../data/cleaned/"
file_name = "jobs_combined.csv"
full_path = os.path.join(path, file_name)

df = pd.read_csv(full_path)

# 2. Drop
df.drop(columns=['data_source'], inplace=True)

# 3. Save
try:
    df.to_csv(full_path, index=False)
    print("✅ data_source dropped!")
    print(f"\n📋 Final columns ({len(df.columns)}):")
    print(df.columns.tolist())
except PermissionError:
    print("❌ Close the CSV file first!")

✅ data_source dropped!

📋 Final columns (14):
['job_title', 'company_name', 'job_type', 'min_experience', 'max_experience', 'experience_level', 'skills_list', 'job_category', 'posted_date', 'job_url', 'city', 'work_type', 'scrape_date', 'company_type']
